# Dynamic mint exposure and recovery

This notebook asks a narrower question than ordinary odor identity classification:

> Does the current Cyranose state indicate **active mint entering now**, **a residual mint response that is recovering**, or **no active mint**?

The spatial application is **Horizontal line raster-454**, whose immutable raw trial ID remains `line_raster_mint_moderate_strip_01` and whose session timestamp is `20260723_165457` (4:54:57 PM).

The Cyranose 320 uses 32 carbon-black/polymer composite chemiresistors. Vapor absorption changes resistance, and a purge is used to return the sensors toward their original resistances. That physical behavior motivates a temporal model rather than a timestamp shift alone. The implementation uses regularized multinomial logistic regression and complete-trial grouped validation, following the official documentation linked below.

Primary technical references:

- [Cyrano Sciences: polymer-composite sensor mechanism and purge behavior](https://cyranosciences.com/technology/tour.html)
- [scikit-learn: regularized multinomial logistic regression](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html)
- [scikit-learn: grouped cross-validation and `StratifiedGroupKFold`](https://scikit-learn.org/stable/modules/cross_validation.html#stratifiedgroupkfold)
- [scikit-learn: nested versus non-nested model selection](https://scikit-learn.org/stable/auto_examples/model_selection/plot_nested_cross_validation_iris.html)

All 0-1 outputs below are **model scores**, not independently calibrated probabilities.

## ELI5: three sensor states

Imagine the 32 sensors as 32 small sponges:

- **Active mint:** the combined 32-sensor pattern is becoming or remaining strongly mint-like while mint is entering.
- **Mint recovery:** active exposure has stopped or decreased, but the sensor pattern is still mint-like and returning toward baseline.
- **No active mint:** the pattern does not support active mint. A separate baseline-distance check determines whether it is also operationally baseline-like.

This is not a rigid one-way sequence. A scan may go `no active mint -> active mint -> recovery -> active mint` whenever the snout meets the source again. Recovery becomes operationally clean only after active-mint evidence is low **and** the 32-sensor change has returned inside the blank-trial baseline envelope.

For sensor vector $r_t$ at time $t$, a finite-difference feature is simply

$$\Delta_h r_t = r_t-r_{t-h}.$$

Positive or negative changes in individual channels are not interpreted alone. The model learns the combined direction across all 32 sensors. Every temporal feature is causal: it uses only the current and earlier readings.

In [101]:
import sys
import site
from pathlib import Path
import json
import hashlib
import platform

USER_SITE = Path(site.getusersitepackages()).resolve()
sys.path[:] = [entry for entry in sys.path if not entry or Path(entry).resolve() != USER_SITE]

import numpy as np
import pandas as pd
import joblib
import sklearn
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import Markdown, display
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import balanced_accuracy_score, confusion_matrix, roc_auc_score
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

def find_repo_root():
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / 'experiments').is_dir() and (candidate / 'record_cyranose_reading_pose.py').exists():
            return candidate
    raise FileNotFoundError('Run this notebook from inside the realsense-apriltag repository.')

ROOT = find_repo_root()
RAW = ROOT / 'experiments' / 'classifier' / 'mint-identity-pilot-01' / 'raw' / 'pcnose'
SENSORS = [f'S{i}' for i in range(1, 33)]
STATE_ORDER = np.array(['active_mint', 'mint_recovery', 'no_active_mint'])
STATE_LABELS = {
    'active_mint': 'Active mint',
    'mint_recovery': 'Mint recovery',
    'no_active_mint': 'No active mint',
}
TRIAL_PATTERNS = {
    'blank': 'blank-*.csv',
    'mint': 'mint-*.csv',
    'bergamot': 'bergamot_??.csv',
    'lemongrass': 'lemongrass_??.csv',
}
FEATURE_CONFIGS = {
    'Current only': {'lookbacks': (), 'ema_taus': ()},
    'Temporal differences (3 s)': {'lookbacks': (0.5, 1.5, 3.0), 'ema_taus': ()},
    'Temporal differences (6 s)': {'lookbacks': (0.5, 1.5, 3.0, 6.0), 'ema_taus': ()},
    'Differences + recovery traces': {
        'lookbacks': (0.5, 1.5, 3.0, 6.0),
        'ema_taus': (1.5, 4.0, 8.0),
    },
}
TEMPORAL_CANDIDATES = [name for name in FEATURE_CONFIGS if name != 'Current only']
C_VALUES = (0.03, 0.1, 0.3)
MIN_HISTORY_S = 6.0
RANDOM_STATE = 23

print('Repository:', ROOT)
print('Stationary PCnose+ data:', RAW.relative_to(ROOT))


Repository: c:\Users\dell-xps\Documents\realsense-apriltag
Stationary PCnose+ data: experiments\classifier\mint-identity-pilot-01\raw\pcnose


In [102]:
def irregular_ema(times, values, tau_s):
    result = np.empty_like(values, dtype=float)
    result[0] = values[0]
    for index in range(1, len(values)):
        dt = max(float(times[index] - times[index - 1]), 0.0)
        alpha = np.exp(-dt / tau_s)
        result[index] = alpha * result[index - 1] + (1.0 - alpha) * values[index]
    return result

def interpolate_vectors(times, values, query_times):
    return np.column_stack([
        np.interp(query_times, times, values[:, sensor_index])
        for sensor_index in range(values.shape[1])
    ])

def build_feature_matrix(cache, config, indices):
    times = cache['times']
    response = cache['response']
    current = response[indices]
    parts = [current]
    for lookback_s in config['lookbacks']:
        previous = interpolate_vectors(times, response, times[indices] - lookback_s)
        parts.append(current - previous)
    for tau_s in config['ema_taus']:
        ema = cache.setdefault('ema', {}).setdefault(
            tau_s, irregular_ema(times, response, tau_s)
        )
        parts.append(current - ema[indices])
    return np.column_stack(parts)

def read_stationary_trial(path, odor_class):
    frame = pd.read_csv(path, skiprows=2)
    times = pd.to_numeric(frame['Time'], errors='coerce').to_numpy(float)
    flags = pd.to_numeric(frame['Flag'], errors='coerce').fillna(-1).astype(int).to_numpy()
    resistances = frame[SENSORS].apply(pd.to_numeric, errors='coerce').to_numpy(float)
    flag1 = resistances[flags == 1]
    if len(flag1) < 4:
        raise ValueError(f'{path.name}: insufficient flag-1 baseline rows')
    baseline = np.median(flag1[len(flag1) // 2:], axis=0)
    response = 100.0 * (resistances - baseline) / baseline
    return {
        'path': path,
        'trial': path.stem,
        'odor_class': odor_class,
        'frame': frame,
        'times': times,
        'flags': flags,
        'response': response,
        'baseline': baseline,
        'ema': {},
    }

trial_cache = {}
feature_parts = {name: [] for name in FEATURE_CONFIGS}
metadata_parts = []

for odor_class, pattern in TRIAL_PATTERNS.items():
    for path in sorted((RAW / odor_class).glob(pattern)):
        cache = read_stationary_trial(path, odor_class)
        trial_cache[cache['trial']] = cache
        flags = cache['flags']
        times = cache['times']
        eligible = np.flatnonzero(
            np.isin(flags, [1, 3, 6, 7])
            & ((times - times[0]) >= MIN_HISTORY_S)
        )
        states = np.full(len(eligible), 'no_active_mint', dtype=object)
        if odor_class == 'mint':
            states[flags[eligible] == 3] = 'active_mint'
            states[np.isin(flags[eligible], [6, 7])] = 'mint_recovery'
        metadata_parts.append(pd.DataFrame({
            'trial': cache['trial'],
            'odor_class': odor_class,
            'row_index': eligible,
            'time_s': times[eligible],
            'elapsed_s': times[eligible] - times[0],
            'flag': flags[eligible],
            'state': states,
            'rms_pct': np.sqrt(np.mean(cache['response'][eligible] ** 2, axis=1)),
        }))
        for name, config in FEATURE_CONFIGS.items():
            feature_parts[name].append(build_feature_matrix(cache, config, eligible))

metadata = pd.concat(metadata_parts, ignore_index=True)
features = {name: np.vstack(parts) for name, parts in feature_parts.items()}
target = metadata['state'].to_numpy()
groups = metadata['trial'].to_numpy()

# Each state receives equal total weight. Within a state, every contributing
# trial receives equal total weight, so long files cannot masquerade as many
# independent experiments.
rows_per_trial_state = metadata.groupby(['state', 'trial']).size().to_dict()
trials_per_state = metadata.groupby('state')['trial'].nunique().to_dict()
sample_weights = np.array([
    1.0 / rows_per_trial_state[(state, trial)] / trials_per_state[state]
    for state, trial in zip(metadata['state'], metadata['trial'])
])
sample_weights *= len(sample_weights) / sample_weights.sum()

assert len(metadata) == len(target) == len(groups)
assert all(matrix.shape[0] == len(metadata) for matrix in features.values())
assert all(np.isfinite(matrix).all() for matrix in features.values())

inventory = metadata.groupby(['odor_class', 'state'])['trial'].nunique().unstack(fill_value=0)
display(inventory)
print('Temporal windows:', len(metadata))
print('Independent files:', metadata['trial'].nunique())
print('Independent mint exposure/recovery files:', metadata.loc[metadata['odor_class'].eq('mint'), 'trial'].nunique())


state,active_mint,mint_recovery,no_active_mint
odor_class,,,
bergamot,0,0,20
blank,0,0,15
lemongrass,0,0,20
mint,15,15,15


Temporal windows: 9688
Independent files: 70
Independent mint exposure/recovery files: 15


## Phase audit: do the files really contain recovery examples?

The training labels come from the device-controlled phase sequence, not from manually inspecting whether a curve looks convenient. In these files, flag 1 is the pre-sample baseline/purge, flag 3 is the sample draw, and flags 6 and 7 occur only after the sample draw as purge/recovery phases.

In [103]:
phase_rows = []
for trial, cache in trial_cache.items():
    flags = cache['flags']
    observed = [int(value) for value in pd.Series(flags).drop_duplicates() if value in (1, 3, 6, 7)]
    phase_rows.append({
        'trial': trial,
        'odor_class': cache['odor_class'],
        'phase sequence': ' -> '.join(map(str, observed)),
        'sequence valid': observed == [1, 3, 6, 7],
    })
phase_audit = pd.DataFrame(phase_rows)
assert phase_audit['sequence valid'].all(), phase_audit.loc[~phase_audit['sequence valid']]
display(phase_audit.groupby('odor_class').agg(
    trials=('trial', 'count'), valid_sequences=('sequence valid', 'sum')
))

representative = trial_cache['mint-1']
representative_rms = np.sqrt(np.mean(representative['response'] ** 2, axis=1))
phase_names = {1: 'Baseline / purge', 3: 'Active mint draw', 6: 'Early recovery', 7: 'Recovery / air purge'}
phase_colors = {1: 'rgba(110,110,110,0.10)', 3: 'rgba(52,168,120,0.15)', 6: 'rgba(220,150,70,0.13)', 7: 'rgba(220,150,70,0.08)'}
phase_figure = go.Figure()
phase_figure.add_trace(go.Scatter(
    x=representative['times'] - representative['times'][0],
    y=representative_rms,
    mode='lines+markers', marker=dict(size=4), name='32-sensor RMS change',
    hovertemplate='t %{x:.1f} s<br>RMS change %{y:.3f}%<extra></extra>',
))
flags = representative['flags']
times = representative['times'] - representative['times'][0]
for flag in [1, 3, 6, 7]:
    positions = np.flatnonzero(flags == flag)
    phase_figure.add_vrect(
        x0=float(times[positions[0]]), x1=float(times[positions[-1]]),
        fillcolor=phase_colors[flag], line_width=0,
        annotation_text=phase_names[flag], annotation_position='top left',
    )
phase_figure.update_xaxes(title='Trial elapsed time (s)')
phase_figure.update_yaxes(title='RMS change from flag-1 baseline (%)')
phase_figure.update_layout(
    title='One mint trial: known exposure and recovery phases',
    height=470, width=1050, margin=dict(l=75, r=35, t=80, b=65),
)
phase_figure.show()


,trials,valid_sequences
odor_class,,
bergamot,20,20
blank,15,15
lemongrass,20,20
mint,15,15


**ELI5.** This plot is one complete movie. The green region tells us when the machine was actively drawing mint. The orange regions begin only after active draw ended, so they provide labeled examples of what the sensor looks like while recovering. A single row is one frame; the entire file is the independent experiment.

## Leakage-safe model selection

The **current-only** model receives one 32-value snapshot. Temporal candidates receive the same snapshot plus exact timestamp-based changes over recent history. The recovery-trace candidate also receives fast and slow exponentially fading memories.

Nested grouped validation is used:

- the inner loop chooses feature history and regularization using training files only;
- the outer loop evaluates that choice on complete untouched files;
- no rows from one file can appear on both sides of a split;
- the spatial raster is not consulted during model selection.

In [104]:
def make_model(c_value):
    return Pipeline([
        ('scale', StandardScaler()),
        ('model', LogisticRegression(
            C=c_value, penalty='l2', solver='lbfgs', max_iter=5000,
        )),
    ])

def aligned_proba(model, x):
    raw = model.predict_proba(x)
    result = np.zeros((len(x), len(STATE_ORDER)), dtype=float)
    for source_column, state in enumerate(model.classes_):
        result[:, np.where(STATE_ORDER == state)[0][0]] = raw[:, source_column]
    return result

def candidate_score(feature_name, c_value, outer_train_index, seed):
    x = features[feature_name]
    local_y = target[outer_train_index]
    local_groups = groups[outer_train_index]
    splitter = StratifiedGroupKFold(n_splits=4, shuffle=True, random_state=seed)
    fold_scores = []
    for inner_train, inner_test in splitter.split(x[outer_train_index], local_y, local_groups):
        train_index = outer_train_index[inner_train]
        test_index = outer_train_index[inner_test]
        assert set(groups[train_index]).isdisjoint(set(groups[test_index]))
        model = make_model(c_value)
        model.fit(
            x[train_index], target[train_index],
            model__sample_weight=sample_weights[train_index],
        )
        prediction = model.predict(x[test_index])
        fold_scores.append(balanced_accuracy_score(
            target[test_index], prediction, sample_weight=sample_weights[test_index],
        ))
    return float(np.mean(fold_scores))

def nested_grouped_oof(candidate_names):
    oof = np.full((len(target), len(STATE_ORDER)), np.nan, dtype=float)
    selections = []
    splitter = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    for fold, (train_index, test_index) in enumerate(
        splitter.split(features['Current only'], target, groups), start=1
    ):
        assert set(groups[train_index]).isdisjoint(set(groups[test_index]))
        candidates = []
        for feature_name in candidate_names:
            for c_value in C_VALUES:
                candidates.append({
                    'feature': feature_name,
                    'C': c_value,
                    'inner balanced accuracy': candidate_score(
                        feature_name, c_value, train_index, RANDOM_STATE + fold,
                    ),
                    'feature count': features[feature_name].shape[1],
                })
        candidate_frame = pd.DataFrame(candidates).sort_values(
            ['inner balanced accuracy', 'feature count', 'C'],
            ascending=[False, True, True],
        )
        selected = candidate_frame.iloc[0]
        model = make_model(float(selected['C']))
        x = features[selected['feature']]
        model.fit(
            x[train_index], target[train_index],
            model__sample_weight=sample_weights[train_index],
        )
        oof[test_index] = aligned_proba(model, x[test_index])
        selections.append({
            'outer fold': fold,
            'selected feature': selected['feature'],
            'selected C': float(selected['C']),
            'inner balanced accuracy': float(selected['inner balanced accuracy']),
            'held-out trials': ', '.join(sorted(set(groups[test_index]))),
        })
    assert np.isfinite(oof).all()
    return oof, pd.DataFrame(selections)

instant_oof, instant_selections = nested_grouped_oof(['Current only'])
temporal_oof, temporal_selections = nested_grouped_oof(TEMPORAL_CANDIDATES)

def validation_metrics(name, probabilities):
    prediction = STATE_ORDER[np.argmax(probabilities, axis=1)]
    return {
        'model': name,
        'balanced accuracy': balanced_accuracy_score(
            target, prediction, sample_weight=sample_weights,
        ),
        'active-mint AUC': roc_auc_score(
            target == 'active_mint', probabilities[:, 0], sample_weight=sample_weights,
        ),
        'recovery AUC': roc_auc_score(
            target == 'mint_recovery', probabilities[:, 1], sample_weight=sample_weights,
        ),
        'active recall': np.average(
            prediction[target == 'active_mint'] == 'active_mint',
            weights=sample_weights[target == 'active_mint'],
        ),
        'recovery recall': np.average(
            prediction[target == 'mint_recovery'] == 'mint_recovery',
            weights=sample_weights[target == 'mint_recovery'],
        ),
        'no-active-mint recall': np.average(
            prediction[target == 'no_active_mint'] == 'no_active_mint',
            weights=sample_weights[target == 'no_active_mint'],
        ),
    }

validation_summary = pd.DataFrame([
    validation_metrics('Current-only snapshot', instant_oof),
    validation_metrics('Temporal history', temporal_oof),
])
display(validation_summary.style.format({column: '{:.3f}' for column in validation_summary if column != 'model'}))
display(temporal_selections[['outer fold', 'selected feature', 'selected C', 'inner balanced accuracy']].style.format({
    'selected C': '{:.2g}', 'inner balanced accuracy': '{:.3f}',
}))


KeyboardInterrupt: 

In [ ]:
instant_prediction = STATE_ORDER[np.argmax(instant_oof, axis=1)]
temporal_prediction = STATE_ORDER[np.argmax(temporal_oof, axis=1)]
instant_matrix = confusion_matrix(target, instant_prediction, labels=STATE_ORDER, normalize='true')
temporal_matrix = confusion_matrix(target, temporal_prediction, labels=STATE_ORDER, normalize='true')

validation_figure = make_subplots(
    rows=1, cols=3,
    subplot_titles=('Held-out balanced accuracy', 'Current-only confusion', 'Temporal confusion'),
    column_widths=[0.25, 0.375, 0.375],
)
validation_figure.add_trace(go.Bar(
    x=validation_summary['model'], y=validation_summary['balanced accuracy'],
    text=validation_summary['balanced accuracy'].map(lambda value: f'{value:.1%}'),
    textposition='outside', marker_color=['#697386', '#536d9e'], showlegend=False,
    hovertemplate='%{x}<br>balanced accuracy %{y:.3f}<extra></extra>',
), row=1, col=1)
for column, matrix in [(2, instant_matrix), (3, temporal_matrix)]:
    validation_figure.add_trace(go.Heatmap(
        z=matrix,
        x=[STATE_LABELS[state] for state in STATE_ORDER],
        y=[STATE_LABELS[state] for state in STATE_ORDER],
        colorscale='Cividis', zmin=0, zmax=1, showscale=(column == 3),
        colorbar=dict(title='Row fraction', len=0.72) if column == 3 else None,
        text=np.vectorize(lambda value: f'{value:.1%}')(matrix), texttemplate='%{text}',
        hovertemplate='true %{y}<br>predicted %{x}<br>fraction %{z:.3f}<extra></extra>',
    ), row=1, col=column)
validation_figure.update_yaxes(range=[0, 1.05], title='Balanced accuracy', row=1, col=1)
validation_figure.update_yaxes(title='True state', autorange='reversed', row=1, col=2)
validation_figure.update_yaxes(title='True state', autorange='reversed', row=1, col=3)
validation_figure.update_xaxes(title='Predicted state', row=1, col=2)
validation_figure.update_xaxes(title='Predicted state', row=1, col=3)
validation_figure.update_layout(height=520, width=1250, margin=dict(l=80, r=80, t=80, b=100))
validation_figure.show()


**ELI5.** The left bar asks whether a short sensor movie beats one photograph on complete unseen files. Each confusion-matrix row is what truly happened; each column is what the model said. A stronger diagonal means fewer active/recovery mix-ups. Because complete files were held out, hundreds of neighboring rows from one recording cannot leak across the train/test boundary.

In [ ]:
# Select the deployment configuration using only grouped stationary-file validation.
full_index = np.arange(len(metadata))
selection_rows = []
for feature_name in TEMPORAL_CANDIDATES:
    for c_value in C_VALUES:
        selection_rows.append({
            'feature': feature_name,
            'C': c_value,
            'grouped balanced accuracy': candidate_score(
                feature_name, c_value, full_index, RANDOM_STATE + 100,
            ),
            'feature count': features[feature_name].shape[1],
        })
selection_table = pd.DataFrame(selection_rows).sort_values(
    ['grouped balanced accuracy', 'feature count', 'C'],
    ascending=[False, True, True],
)
selected_feature = str(selection_table.iloc[0]['feature'])
selected_c = float(selection_table.iloc[0]['C'])
temporal_model = make_model(selected_c)
temporal_model.fit(
    features[selected_feature], target,
    model__sample_weight=sample_weights,
)

# Operational baseline-like limit: 95th percentile of known blank sample draws.
baseline_rms_limit = float(metadata.loc[
    metadata['odor_class'].eq('blank') & metadata['flag'].eq(3), 'rms_pct'
].quantile(0.95))

display(selection_table.style.format({
    'C': '{:.2g}', 'grouped balanced accuracy': '{:.3f}',
}))
print('Locked temporal feature set:', selected_feature)
print('Locked regularization C:', selected_c)
print(f'Operational blank RMS envelope (95th percentile): {baseline_rms_limit:.4f}%')


,feature,C,grouped balanced accuracy,feature count
8,Differences + recovery traces,0.3,0.897,256
7,Differences + recovery traces,0.1,0.888,256
5,Temporal differences (6 s),0.3,0.877,160
2,Temporal differences (3 s),0.3,0.873,128
6,Differences + recovery traces,0.03,0.870,256
4,Temporal differences (6 s),0.1,0.859,160
1,Temporal differences (3 s),0.1,0.853,128
3,Temporal differences (6 s),0.03,0.838,160
0,Temporal differences (3 s),0.03,0.832,128


Locked temporal feature set: Differences + recovery traces
Locked regularization C: 0.3
Operational blank RMS envelope (95th percentile): 0.0502%


**ELI5.** History length is the length of the movie clip. Regularization is a "do not memorize weird wiggles" knob. The table chooses both using only stationary PCnose+ files. Horizontal line raster-454 has still not been used, so its known strip location cannot influence this decision.

The baseline envelope is separate from the three-state classifier. `No active mint` means no active mint identity was inferred; `baseline-like` additionally requires the overall 32-sensor change to fall inside the range observed during blank sample draws.

In [ ]:
# Reproduce notebook 05's original two-stage current-state classifier so the
# before/after spatial figures use the same pre-temporal score.
original_records = []
for trial, cache in trial_cache.items():
    sample = cache['response'][cache['flags'] == 3]
    smoothed = pd.DataFrame(sample).rolling(3, min_periods=1).median().to_numpy()
    for sample_index, vector in enumerate(smoothed):
        original_records.append({
            'trial': trial,
            'odor_class': cache['odor_class'],
            'sample_index': sample_index,
            'vector': vector,
        })
original_rows = pd.DataFrame(original_records)

def equal_binary_trial_weights(frame, binary_target):
    per_trial_rows = frame.groupby('trial').size().to_dict()
    trial_targets = pd.DataFrame({
        'trial': frame['trial'].to_numpy(), 'target': binary_target,
    }).drop_duplicates()
    trials_per_class = trial_targets.groupby('target')['trial'].nunique().to_dict()
    weights = np.array([
        1.0 / per_trial_rows[trial] / (2.0 * trials_per_class[label])
        for trial, label in zip(frame['trial'], binary_target)
    ])
    return weights * len(weights) / weights.sum()

def make_binary_model():
    return Pipeline([
        ('scale', StandardScaler()),
        ('model', LogisticRegression(C=0.1, penalty='l2', solver='lbfgs', max_iter=5000)),
    ])

presence_x = np.vstack(original_rows['vector'])
presence_y = original_rows['odor_class'].ne('blank').astype(int).to_numpy()
presence_weights = equal_binary_trial_weights(original_rows, presence_y)
presence_model = make_binary_model()
presence_model.fit(presence_x, presence_y, model__sample_weight=presence_weights)

identity_rows = original_rows.loc[
    original_rows['odor_class'].ne('blank') & original_rows['sample_index'].ge(29)
].copy()
identity_raw = np.vstack(identity_rows['vector'])
identity_x = identity_raw / np.maximum(np.linalg.norm(identity_raw, axis=1, keepdims=True), 1e-12)
identity_y = identity_rows['odor_class'].eq('mint').astype(int).to_numpy()
identity_weights = equal_binary_trial_weights(identity_rows, identity_y)
identity_model = make_binary_model()
identity_model.fit(identity_x, identity_y, model__sample_weight=identity_weights)
print('Reproduced notebook 05 current-state classifier.')


Reproduced notebook 05 current-state classifier.


## Apply the locked model to Horizontal line raster-454

The spatial recording is not used for training or hyperparameter selection. Both before/after maps use the same 2.0 s working pose shift, reviewed raster window, 1.0-3.5 cm height band, display orientation, and approximate 17 x 3 cm source footprint. The 2.0 s value is a conservative engineering compromise after the lag audit: it lies between the visually plausible 1.5 and 2.25 s cases, was directly tested, and avoids false precision while remaining near the numerical optimum. The reported source coordinates are held fixed so changing lag cannot move the ground-truth target. Physical history is calculated from the recorder's host elapsed time; the legacy device clock is not treated as wall-clock time.

In [ ]:
RASTER_TRIAL_ID = 'line_raster_mint_moderate_strip_01'
RASTER_DISPLAY_NAME = 'Horizontal line raster-454'
RESPONSE_LAG_S = 2.0
RASTER_WINDOW_S = (45.0, 132.0)
HEIGHT_BAND_CM = (1.0, 3.5)
BOUNDS = (10.0, 40.0, 19.0, 47.0)
SOURCE_LENGTH_CM = 17.0
SOURCE_WIDTH_CM = 3.0
REPORTED_SOURCE_CENTER_X_CM = 25.355350731
REPORTED_SOURCE_CENTER_Y_CM = 33.807212754
MAX_INTERPOLATION_GAP_S = 1.5
ODOR_PRESENCE_GATE_THRESHOLD = 0.5

def find_session(trial_id):
    matches = []
    for metadata_path in ROOT.rglob('session_metadata.json'):
        payload = json.loads(metadata_path.read_text(encoding='utf-8'))
        if payload.get('trial_id') == trial_id:
            matches.append(metadata_path.parent)
    if len(matches) != 1:
        raise RuntimeError(f'Expected one session for {trial_id!r}; found {matches}')
    return matches[0]

raster_session = find_session(RASTER_TRIAL_ID)
raster_metadata = json.loads((raster_session / 'session_metadata.json').read_text(encoding='utf-8'))
raster_alignment = json.loads((raster_session / 'alignment_summary.json').read_text(encoding='utf-8'))
assert raster_metadata.get('display_name') == RASTER_DISPLAY_NAME

raster_frame = pd.read_csv(raster_session / 'cyranose_reading_pose.csv')
# Use host elapsed seconds for physical history. The legacy Cyranose device
# clock in direct-serial sessions does not advance at wall-clock speed.
raster_times = pd.to_numeric(raster_frame['pcnose_elapsed_s'], errors='coerce').to_numpy(float)
raster_flags = pd.to_numeric(raster_frame['pcnose_flag'], errors='coerce').fillna(-1).astype(int).to_numpy()
raster_sensor_columns = [f'pcnose_S{i}_kohm' for i in range(1, 33)]
raster_resistances = raster_frame[raster_sensor_columns].apply(pd.to_numeric, errors='coerce').to_numpy(float)
raster_flag1 = raster_resistances[raster_flags == 1]
raster_baseline = np.median(raster_flag1[len(raster_flag1) // 2:], axis=0)
raster_response = 100.0 * (raster_resistances - raster_baseline) / raster_baseline
raster_cache = {
    'times': raster_times, 'flags': raster_flags, 'response': raster_response, 'ema': {},
}
scan_indices_all = np.flatnonzero(raster_flags == 2)
scan_indices = scan_indices_all[(raster_times[scan_indices_all] - raster_times[0]) >= MIN_HISTORY_S]

dynamic_x = build_feature_matrix(raster_cache, FEATURE_CONFIGS[selected_feature], scan_indices)
dynamic_scores = aligned_proba(temporal_model, dynamic_x)

original_scan_response = raster_response[scan_indices_all]
original_scan_features = pd.DataFrame(original_scan_response).rolling(3, min_periods=1).median().to_numpy()
original_presence = presence_model.predict_proba(original_scan_features)[:, 1]
original_direction = original_scan_features / np.maximum(
    np.linalg.norm(original_scan_features, axis=1, keepdims=True), 1e-12,
)
original_identity = identity_model.predict_proba(original_direction)[:, 1]
original_presence_lookup = dict(zip(scan_indices_all, original_presence))
original_identity_lookup = dict(zip(scan_indices_all, original_identity))
original_score_lookup = dict(zip(scan_indices_all, original_presence * original_identity))

scored = raster_frame.iloc[scan_indices].copy()
scored['odor_presence_score'] = [original_presence_lookup[index] for index in scan_indices]
scored['mint_identity_score'] = [original_identity_lookup[index] for index in scan_indices]
scored['original_mint_score'] = [original_score_lookup[index] for index in scan_indices]
for column, state in enumerate(STATE_ORDER):
    scored[f'{state}_score'] = dynamic_scores[:, column]
scored['rms_pct'] = np.sqrt(np.mean(raster_response[scan_indices] ** 2, axis=1))
scored['baseline_like'] = scored['rms_pct'] <= baseline_rms_limit

pose = raster_frame.dropna(subset=[
    'pose_elapsed_s', 'snout_desk_x_cm', 'snout_desk_y_cm', 'snout_desk_z_cm',
]).copy()
pose_times = pose['pose_elapsed_s'].to_numpy(float)
pose_xyz = pose[['snout_desk_x_cm', 'snout_desk_y_cm', 'snout_desk_z_cm']].to_numpy(float)
xmin, xmax, ymin, ymax = BOUNDS
aligned_rows = []
for _, reading in scored.iterrows():
    if pd.isna(reading.get('pose_elapsed_s')):
        continue
    corrected_time = float(reading['pose_elapsed_s']) - RESPONSE_LAG_S
    index = int(np.searchsorted(pose_times, corrected_time))
    if index == 0 or index == len(pose_times):
        continue
    gap = pose_times[index] - pose_times[index - 1]
    if gap <= 0 or gap > MAX_INTERPOLATION_GAP_S:
        continue
    fraction = (corrected_time - pose_times[index - 1]) / gap
    x, y, z = pose_xyz[index - 1] + fraction * (pose_xyz[index] - pose_xyz[index - 1])
    if not (xmin <= x <= xmax and ymin <= y <= ymax):
        continue
    height = abs(z)
    aligned_rows.append({
        'elapsed_s': float(reading['pose_elapsed_s']),
        'x': x, 'y': y, 'z': z, 'height': height,
        'height_valid': HEIGHT_BAND_CM[0] <= height <= HEIGHT_BAND_CM[1],
        'odor_presence_score': float(reading['odor_presence_score']),
        'mint_identity_score': float(reading['mint_identity_score']),
        'original_mint_score': float(reading['original_mint_score']),
        'active_mint_score': float(reading['active_mint_score']),
        'mint_recovery_score': float(reading['mint_recovery_score']),
        'no_active_mint_score': float(reading['no_active_mint_score']),
        'rms_pct': float(reading['rms_pct']),
        'baseline_like': bool(reading['baseline_like']),
    })

raster_all = pd.DataFrame(aligned_rows)
raster_all['presence_gate_pass'] = (
    raster_all['odor_presence_score'] >= ODOR_PRESENCE_GATE_THRESHOLD
)
raster_all['active_mint_gated_score'] = np.where(
    raster_all['presence_gate_pass'], raster_all['active_mint_score'], 0.0,
)
raster_window = raster_all[raster_all['elapsed_s'].between(*RASTER_WINDOW_S)].copy()
raster_qc = raster_window[raster_window['height_valid']].copy()
source_center_x = REPORTED_SOURCE_CENTER_X_CM
source_center_y = REPORTED_SOURCE_CENTER_Y_CM
source_x0 = source_center_x - SOURCE_LENGTH_CM / 2
source_x1 = source_center_x + SOURCE_LENGTH_CM / 2
source_y0 = source_center_y - SOURCE_WIDTH_CM / 2
source_y1 = source_center_y + SOURCE_WIDTH_CM / 2
for frame in [raster_window, raster_qc]:
    frame['source_region'] = (
        frame['x'].between(source_x0, source_x1)
        & frame['y'].between(source_y0, source_y1)
    )

print('Session:', raster_session.relative_to(ROOT))
print(
    f"Digital alignment: {raster_alignment['matched_readings']}/"
    f"{raster_alignment['total_readings']} within 250 ms; "
    f"p95 {raster_alignment['absolute_pose_minus_pcnose_ms']['p95']:.1f} ms"
)
print(f'Reviewed raster rows: {len(raster_window)}; height-qualified: {len(raster_qc)}')


Session: experiments\spatial-mapping\line-raster\horizontal-line-raster-454\cyranose_reading_pose_session_20260723_165457
Digital alignment: 238/238 within 250 ms; p95 95.2 ms
Reviewed raster rows: 157; height-qualified: 97


In [ ]:
def add_source_overlay(figure, row=1, col=1):
    figure.add_shape(
        type='circle', x0=source_x0, x1=source_x1, y0=source_y0, y1=source_y1,
        fillcolor='rgba(63, 191, 143, 0.20)',
        line=dict(color='rgba(25, 130, 92, 0.95)', width=2, dash='dash'),
        layer='above', row=row, col=col,
    )
    figure.add_annotation(
        x=source_center_x, y=source_y1,
        text='Reported 17 x 3 cm strip (approx. location)',
        showarrow=False, yshift=12, font=dict(color='rgb(25, 110, 80)', size=11),
        row=row, col=col,
    )

def add_spatial_score(figure, score_column, colorbar_title):
    figure.add_trace(go.Scatter(
        x=raster_window['x'], y=raster_window['y'], mode='lines',
        line=dict(color='rgba(150,150,150,0.65)', width=1), showlegend=False, hoverinfo='skip',
    ), row=1, col=1)
    for valid, symbol, label in [
        (True, 'circle', 'height 1.0-3.5 cm'),
        (False, 'x', 'outside height band'),
    ]:
        subset = raster_window[raster_window['height_valid'].eq(valid)]
        figure.add_trace(go.Scatter(
            x=subset['x'], y=subset['y'], mode='markers', name=label,
            marker=dict(
                size=8, symbol=symbol, color=subset[score_column], colorscale='Cividis',
                cmin=0, cmax=1, showscale=valid,
                colorbar=dict(title=colorbar_title, len=0.78, x=0.46) if valid else None,
            ),
            customdata=np.column_stack([
                subset['height'], subset['odor_presence_score'],
                subset['active_mint_score'], subset['active_mint_gated_score'],
                subset['mint_recovery_score'], subset['no_active_mint_score'],
            ]),
            hovertemplate=(
                'x %{x:.1f} cm<br>y %{y:.1f} cm<br>displayed score %{marker.color:.3f}'
                '<br>height %{customdata[0]:.2f} cm<br>odor presence %{customdata[1]:.3f}'
                '<br>raw temporal active %{customdata[2]:.3f}'
                '<br>presence-gated active %{customdata[3]:.3f}'
                '<br>recovery %{customdata[4]:.3f}<br>no active mint %{customdata[5]:.3f}'
                '<extra></extra>'
            ),
        ), row=1, col=1)
    for point, symbol, label, position in [
        (raster_window.iloc[0], 'triangle-up', 'Start', 'bottom center'),
        (raster_window.iloc[-1], 'square', 'End', 'top center'),
    ]:
        figure.add_trace(go.Scatter(
            x=[point['x']], y=[point['y']], mode='markers+text',
            marker=dict(size=12, symbol=symbol, color='crimson'),
            text=[label], textposition=position, showlegend=False,
        ), row=1, col=1)
    add_source_overlay(figure)
    figure.update_xaxes(title_text='Paper horizontal / Desk X (cm)', autorange='reversed', row=1, col=1)
    figure.update_yaxes(
        title_text='Paper vertical / Desk Y (cm)', autorange='reversed',
        scaleanchor='x', scaleratio=1, row=1, col=1,
    )


In [ ]:
ACTIVE_MINT_COLOR = '#59c3e1'
RECOVERY_COLOR = '#78c98d'
SNOUT_HEIGHT_COLOR = '#e76f92'

before_figure = make_subplots(
    rows=1, cols=2, specs=[[{}, {'secondary_y': True}]],
    subplot_titles=(
        'Original classifier scores on the tracked raster',
        'Original classifier response and snout height over time',
    ), horizontal_spacing=0.12,
)
add_spatial_score(before_figure, 'original_mint_score', 'Original mint evidence')
before_figure.add_trace(go.Scatter(
    x=raster_all['elapsed_s'], y=raster_all['original_mint_score'], mode='lines+markers',
    name='Original mint score', marker=dict(size=4, color=ACTIVE_MINT_COLOR),
    line=dict(width=3, color=ACTIVE_MINT_COLOR),
), row=1, col=2, secondary_y=False)
before_figure.add_trace(go.Scatter(
    x=raster_all['elapsed_s'], y=raster_all['height'], mode='lines',
    name='Snout height', line=dict(width=2, color=SNOUT_HEIGHT_COLOR),
), row=1, col=2, secondary_y=True)
before_figure.add_vrect(
    x0=RASTER_WINDOW_S[0], x1=RASTER_WINDOW_S[1], fillcolor='rgba(100,100,100,0.08)',
    line_width=0, annotation_text='reviewed raster window', annotation_position='top left',
    row=1, col=2,
)
before_figure.add_hline(y=0.5, line_dash='dash', line_width=1, row=1, col=2, secondary_y=False)
before_figure.add_hline(y=HEIGHT_BAND_CM[0], line_dash='dot', line_width=1, row=1, col=2, secondary_y=True)
before_figure.add_hline(y=HEIGHT_BAND_CM[1], line_dash='dot', line_width=1, row=1, col=2, secondary_y=True)
before_figure.update_xaxes(title_text='Host elapsed time (s)', row=1, col=2)
before_figure.update_yaxes(title_text='Original mint evidence (0-1)', range=[-0.03, 1.03], row=1, col=2, secondary_y=False)
before_figure.update_yaxes(title_text='Snout height (cm)', row=1, col=2, secondary_y=True)
before_figure.update_layout(
    title=f'{RASTER_DISPLAY_NAME} | before temporal logic',
    height=650, width=1250, margin=dict(l=75, r=90, t=100, b=65),
    legend=dict(orientation='h', yanchor='bottom', y=1.04, xanchor='right', x=1),
)
before_figure.show()


**ELI5.** This is the old logic. It sees one mint-like sensor state at a time. A reading can remain yellow after leaving the strip because the model has no concept of "I am drying out from the previous mint exposure." The 2.0 s working pose shift moves the response backward in time but cannot shorten its recovery tail.

In [ ]:
after_figure = make_subplots(
    rows=2, cols=2,
    specs=[[{'rowspan': 2}, {}], [None, {}]],
    subplot_titles=(
        'Temporal active-mint evidence on the tracked raster',
        'Active and recovery evidence over time',
        'Snout height quality control',
    ),
    column_widths=[0.46, 0.54], row_heights=[0.64, 0.36],
    horizontal_spacing=0.13, vertical_spacing=0.13,
)
add_spatial_score(
    after_figure, 'active_mint_gated_score', 'Presence-gated active-mint evidence'
)
after_figure.add_trace(go.Scatter(
    x=raster_all['elapsed_s'], y=raster_all['active_mint_gated_score'], mode='lines+markers',
    name='Presence-gated active mint now', marker=dict(size=4, color='#238b78'),
    line=dict(width=3, color='#238b78'),
), row=1, col=2)
after_figure.add_trace(go.Scatter(
    x=raster_all['elapsed_s'], y=raster_all['mint_recovery_score'], mode='lines',
    name='Recovering from mint', line=dict(width=3, color='#d97732'),
), row=1, col=2)
after_figure.add_hline(y=0.5, line_dash='dot', line_width=1, row=1, col=2)
after_figure.add_trace(go.Scatter(
    x=raster_all['elapsed_s'], y=raster_all['height'], mode='lines+markers',
    name='Snout height', marker=dict(size=3, color='#53657a'),
    line=dict(width=2, color='#53657a'),
), row=2, col=2)
height_outside = raster_all[
    ~raster_all['height'].between(HEIGHT_BAND_CM[0], HEIGHT_BAND_CM[1])
]
after_figure.add_trace(go.Scatter(
    x=height_outside['elapsed_s'], y=height_outside['height'], mode='markers',
    name='Height outside accepted band', marker=dict(size=7, color='#b23a48', symbol='x'),
), row=2, col=2)
after_figure.add_hrect(
    y0=HEIGHT_BAND_CM[0], y1=HEIGHT_BAND_CM[1],
    fillcolor='rgba(58, 145, 112, 0.13)', line_width=0,
    annotation_text='accepted 1.0-3.5 cm band', annotation_position='top left',
    row=2, col=2,
)
for plot_row in (1, 2):
    after_figure.add_vrect(
        x0=RASTER_WINDOW_S[0], x1=RASTER_WINDOW_S[1],
        fillcolor='rgba(100,100,100,0.06)', line_width=0,
        row=plot_row, col=2,
    )
after_figure.update_xaxes(title_text='Host elapsed time (s)', row=2, col=2)
after_figure.update_yaxes(
    title_text='Temporal state evidence (0-1)', range=[-0.03, 1.03], row=1, col=2,
)
after_figure.update_yaxes(title_text='Snout height (cm)', row=2, col=2)
after_figure.update_layout(
    title=f'{RASTER_DISPLAY_NAME} | after temporal logic',
    height=760, width=1300, margin=dict(l=75, r=95, t=115, b=65),
    legend=dict(orientation='h', yanchor='bottom', y=1.04, xanchor='right', x=1),
)
# Superseded by the matched presentation chart below.

# Matched spatial heatmaps. All three panels use exactly the same retained
# readings, grid, support rule, and color scale; only the model evidence changes.
HEATMAP_GRID_X = np.arange(10.0, 40.01, 1.0)
HEATMAP_GRID_Y = np.arange(19.0, 47.01, 1.0)
HEATMAP_RADIUS_CM = 3.0
HEATMAP_SIGMA_CM = 1.7

def smooth_evidence_grid(frame, score_column):
    point_x = frame['x'].to_numpy(float)
    point_y = frame['y'].to_numpy(float)
    point_score = frame[score_column].to_numpy(float)
    grid = []
    for grid_y in HEATMAP_GRID_Y:
        row_values = []
        for grid_x in HEATMAP_GRID_X:
            distance_sq = (point_x - grid_x) ** 2 + (point_y - grid_y) ** 2
            nearby = distance_sq <= HEATMAP_RADIUS_CM ** 2
            if nearby.sum() < 2:
                row_values.append(None)
                continue
            weights = np.exp(-distance_sq[nearby] / (2 * HEATMAP_SIGMA_CM ** 2))
            row_values.append(float(np.average(point_score[nearby], weights=weights)))
        grid.append(row_values)
    return grid

evidence_heatmaps = {
    'Before temporal logic': smooth_evidence_grid(raster_qc, 'original_mint_score'),
    'Presence-gated active mint now': smooth_evidence_grid(
        raster_qc, 'active_mint_gated_score'
    ),
    'Recovering from mint': smooth_evidence_grid(raster_qc, 'mint_recovery_score'),
}
heatmap_figure = make_subplots(
    rows=1, cols=3,
    subplot_titles=tuple(evidence_heatmaps),
    horizontal_spacing=0.055,
)
for column, (label, grid) in enumerate(evidence_heatmaps.items(), start=1):
    heatmap_figure.add_trace(go.Heatmap(
        x=HEATMAP_GRID_X, y=HEATMAP_GRID_Y, z=grid,
        coloraxis='coloraxis', connectgaps=False,
        hovertemplate=(
            'Desk X %{x:.1f} cm<br>Desk Y %{y:.1f} cm'
            f'<br>{label}: %{{z:.3f}}<extra></extra>'
        ),
    ), row=1, col=column)
    add_source_overlay(heatmap_figure, row=1, col=column)
    heatmap_figure.update_xaxes(
        title_text='Desk X (cm)', autorange='reversed', row=1, col=column,
    )
heatmap_figure.update_yaxes(
    title_text='Desk Y (cm)', autorange='reversed',
    scaleanchor='x', scaleratio=1, row=1, col=1,
)
heatmap_figure.update_yaxes(
    autorange='reversed', scaleanchor='x2', scaleratio=1, row=1, col=2,
)
heatmap_figure.update_yaxes(
    autorange='reversed', scaleanchor='x3', scaleratio=1, row=1, col=3,
)
heatmap_figure.update_layout(
    title=(
        f'{RASTER_DISPLAY_NAME} | before versus active versus recovery | '
        'same retained readings and 3 cm smoothing'
    ),
    width=1450, height=620, template='plotly_white', showlegend=False,
    coloraxis=dict(
        colorscale='Cividis', cmin=0, cmax=1,
        colorbar=dict(title='Model evidence<br>(0-1)', x=1.01),
    ),
    margin=dict(l=65, r=145, t=100, b=65),
)
# Retained as a diagnostic calculation; the standalone presentation heatmaps are shown below.
supported_cells = sum(
    value is not None for row_values in evidence_heatmaps['Before temporal logic']
    for value in row_values
)
print(
    f'Heatmap support: {supported_cells}/'
    f'{len(HEATMAP_GRID_X) * len(HEATMAP_GRID_Y)} grid cells; '
    'white means fewer than two retained readings within 3 cm.'
)

# Balanced presentation view. Keep the strict QC map above, but recover
# near-threshold measurements with a continuous height reliability weight.
# This avoids widening the spatial radius (which would blur the field) and
# avoids accepting a cell from only one reading.
HEIGHT_TAPER_FLOOR_CM = 0.5
MIN_HEIGHT_WEIGHTED_SUPPORT = 1.25

def height_reliability(height):
    height = np.asarray(height, dtype=float)
    reliability = np.zeros_like(height)
    reliability[(height >= HEIGHT_BAND_CM[0]) & (height <= HEIGHT_BAND_CM[1])] = 1.0
    near_surface = (height >= HEIGHT_TAPER_FLOOR_CM) & (height < HEIGHT_BAND_CM[0])
    reliability[near_surface] = (
        (height[near_surface] - HEIGHT_TAPER_FLOOR_CM)
        / (HEIGHT_BAND_CM[0] - HEIGHT_TAPER_FLOOR_CM)
    )
    return np.clip(reliability, 0.0, 1.0)

def smooth_height_weighted_grid(frame, score_column):
    point_x = frame['x'].to_numpy(float)
    point_y = frame['y'].to_numpy(float)
    point_score = frame[score_column].to_numpy(float)
    point_height_weight = height_reliability(frame['height'])
    score_grid, support_grid = [], []
    for grid_y in HEATMAP_GRID_Y:
        score_row, support_row = [], []
        for grid_x in HEATMAP_GRID_X:
            distance_sq = (point_x - grid_x) ** 2 + (point_y - grid_y) ** 2
            nearby = (
                (distance_sq <= HEATMAP_RADIUS_CM ** 2)
                & (point_height_weight > 0)
            )
            effective_height_support = float(point_height_weight[nearby].sum())
            if nearby.sum() < 2 or effective_height_support < MIN_HEIGHT_WEIGHTED_SUPPORT:
                score_row.append(None)
                support_row.append(None)
                continue
            spatial_weight = np.exp(
                -distance_sq[nearby] / (2 * HEATMAP_SIGMA_CM ** 2)
            )
            combined_weight = spatial_weight * point_height_weight[nearby]
            score_row.append(float(np.average(point_score[nearby], weights=combined_weight)))
            support_row.append(effective_height_support)
        score_grid.append(score_row)
        support_grid.append(support_row)
    return score_grid, support_grid

weighted_active_grid, weighted_support_grid = smooth_height_weighted_grid(
    raster_window, 'active_mint_gated_score'
)
balanced_figure = make_subplots(
    rows=1, cols=3,
    subplot_titles=(
        'Strict QC: 1.0-3.5 cm only',
        'Balanced view: near-height readings softly weighted',
        'Height-weighted nearby measurement support',
    ),
    horizontal_spacing=0.065,
)
for column, (label, grid) in enumerate([
    ('Strict presence-gated active-mint evidence',
     evidence_heatmaps['Presence-gated active mint now']),
    ('Height-weighted active-mint evidence', weighted_active_grid),
], start=1):
    balanced_figure.add_trace(go.Heatmap(
        x=HEATMAP_GRID_X, y=HEATMAP_GRID_Y, z=grid,
        coloraxis='coloraxis', connectgaps=False,
        hovertemplate=(
            'Desk X %{x:.1f} cm<br>Desk Y %{y:.1f} cm'
            f'<br>{label}: %{{z:.3f}}<extra></extra>'
        ),
    ), row=1, col=column)
balanced_figure.add_trace(go.Heatmap(
    x=HEATMAP_GRID_X, y=HEATMAP_GRID_Y, z=weighted_support_grid,
    coloraxis='coloraxis2', connectgaps=False,
    hovertemplate=(
        'Desk X %{x:.1f} cm<br>Desk Y %{y:.1f} cm'
        '<br>Height-weighted nearby readings: %{z:.2f}<extra></extra>'
    ),
), row=1, col=3)
for column in range(1, 4):
    add_source_overlay(balanced_figure, row=1, col=column)
    balanced_figure.update_xaxes(
        title_text='Desk X (cm)', autorange='reversed', row=1, col=column,
    )
    balanced_figure.update_yaxes(
        title_text='Desk Y (cm)' if column == 1 else None,
        autorange='reversed', scaleanchor=f'x{column if column > 1 else ""}',
        scaleratio=1, row=1, col=column,
    )
weighted_supported_cells = sum(
    value is not None for row_values in weighted_active_grid for value in row_values
)
support_values = np.array([
    value for row_values in weighted_support_grid for value in row_values
    if value is not None
], dtype=float)
balanced_figure.update_layout(
    title=(
        f'{RASTER_DISPLAY_NAME} | strict QC versus height-weighted presentation view'
    ),
    width=1500, height=620, template='plotly_white', showlegend=False,
    coloraxis=dict(
        colorscale='Cividis', cmin=0, cmax=1,
        colorbar=dict(title='Active-mint<br>evidence (0-1)', x=0.675, len=0.82),
    ),
    coloraxis2=dict(
        colorscale='Blues', cmin=MIN_HEIGHT_WEIGHTED_SUPPORT,
        cmax=float(np.percentile(support_values, 95)),
        colorbar=dict(title='Height-weighted<br>nearby readings', x=1.04, len=0.82),
    ),
    margin=dict(l=65, r=220, t=110, b=65),
)
# Retained as a support audit; the four-chart presentation sequence is shown below.

below_strict = raster_window['height'] < HEIGHT_BAND_CM[0]
near_strict = raster_window['height'].between(0.75, HEIGHT_BAND_CM[0], inclusive='left')
above_strict = raster_window['height'] > HEIGHT_BAND_CM[1]
print(
    f'Height audit: {below_strict.sum()} below {HEIGHT_BAND_CM[0]:.1f} cm '
    f'({near_strict.sum()} between 0.75 and 1.0 cm); '
    f'{above_strict.sum()} above {HEIGHT_BAND_CM[1]:.1f} cm.'
)
print(
    f'Balanced support: {weighted_supported_cells}/'
    f'{len(HEATMAP_GRID_X) * len(HEATMAP_GRID_Y)} grid cells '
    f'({weighted_supported_cells / (len(HEATMAP_GRID_X) * len(HEATMAP_GRID_Y)):.1%}); '
    f'strict support was {supported_cells / (len(HEATMAP_GRID_X) * len(HEATMAP_GRID_Y)):.1%}.'
)

# Presentation chart 2: visually matched to the before-temporal chart.
presentation_after_figure = make_subplots(
    rows=1, cols=2, specs=[[{}, {'secondary_y': True}]],
    subplot_titles=(
        'Presence-gated active-mint scores on the tracked raster',
        'Temporal state evidence and snout height over time',
    ), horizontal_spacing=0.12,
)
add_spatial_score(
    presentation_after_figure, 'active_mint_gated_score',
    'Presence-gated active-mint evidence'
)
presentation_after_figure.add_trace(go.Scatter(
    x=raster_all['elapsed_s'], y=raster_all['active_mint_gated_score'],
    mode='lines+markers', name='Presence-gated active mint now',
    marker=dict(size=4, color=ACTIVE_MINT_COLOR, symbol='circle'),
    line=dict(width=3, color=ACTIVE_MINT_COLOR),
), row=1, col=2, secondary_y=False)
presentation_after_figure.add_trace(go.Scatter(
    x=raster_all['elapsed_s'], y=raster_all['mint_recovery_score'],
    mode='lines+markers', name='Mint decreasing / recovery',
    marker=dict(size=4, color=RECOVERY_COLOR, symbol='diamond'),
    line=dict(width=3, color=RECOVERY_COLOR),
), row=1, col=2, secondary_y=False)
presentation_after_figure.add_trace(go.Scatter(
    x=raster_all['elapsed_s'], y=raster_all['height'],
    mode='lines', name='Snout height',
    line=dict(width=2, color=SNOUT_HEIGHT_COLOR),
), row=1, col=2, secondary_y=True)
presentation_after_figure.add_vrect(
    x0=RASTER_WINDOW_S[0], x1=RASTER_WINDOW_S[1],
    fillcolor='rgba(100,100,100,0.08)', line_width=0,
    annotation_text='reviewed raster window', annotation_position='top left',
    row=1, col=2,
)
presentation_after_figure.add_hline(
    y=0.5, line_dash='dash', line_width=1,
    row=1, col=2, secondary_y=False,
)
for height_limit in HEIGHT_BAND_CM:
    presentation_after_figure.add_hline(
        y=height_limit, line_dash='dot', line_width=1,
        row=1, col=2, secondary_y=True,
    )
presentation_after_figure.update_xaxes(title_text='Host elapsed time (s)', row=1, col=2)
presentation_after_figure.update_yaxes(
    title_text='Temporal state evidence (0-1)', range=[-0.03, 1.03],
    row=1, col=2, secondary_y=False,
)
presentation_after_figure.update_yaxes(
    title_text='Snout height (cm)', row=1, col=2, secondary_y=True,
)
presentation_after_figure.update_layout(
    title=f'{RASTER_DISPLAY_NAME} | after temporal logic',
    height=650, width=1250, margin=dict(l=75, r=90, t=100, b=65),
    legend=dict(orientation='h', yanchor='bottom', y=1.04, xanchor='right', x=1),
)
presentation_after_figure.show()

# Presentation charts 3 and 4: same grid, support, soft height weighting,
# pose correction, radius, and color scale so the visual comparison is fair.
weighted_before_grid, weighted_before_support = smooth_height_weighted_grid(
    raster_window, 'original_mint_score'
)

def presentation_heatmap(grid, title, colorbar_title, detection_threshold=None):
    figure = make_subplots(rows=1, cols=1)
    figure.add_trace(go.Heatmap(
        x=HEATMAP_GRID_X, y=HEATMAP_GRID_Y, z=grid,
        colorscale='Cividis', zmin=0, zmax=1, connectgaps=False,
        colorbar=dict(title=colorbar_title, len=0.82),
        hovertemplate=(
            'Desk X %{x:.1f} cm<br>Desk Y %{y:.1f} cm'
            '<br>Model evidence %{z:.3f}<extra></extra>'
        ),
    ), row=1, col=1)
    if detection_threshold is not None:
        figure.add_trace(go.Contour(
            x=HEATMAP_GRID_X, y=HEATMAP_GRID_Y, z=grid,
            contours=dict(
                start=detection_threshold, end=detection_threshold, size=1,
                coloring='none', showlabels=True,
            ),
            line=dict(color='#394150', width=2, dash='dash'),
            showscale=False, name=f'{detection_threshold:.1f} working detection boundary',
            hoverinfo='skip',
        ), row=1, col=1)
    add_source_overlay(figure, row=1, col=1)
    figure.update_xaxes(title_text='Desk X (cm)', autorange='reversed', row=1, col=1)
    figure.update_yaxes(
        title_text='Desk Y (cm)', autorange='reversed',
        scaleanchor='x', scaleratio=1, row=1, col=1,
    )
    figure.add_annotation(
        x=0, y=-0.13, xref='paper', yref='paper', xanchor='left',
        text=(
            f'Soft height weighting | {weighted_supported_cells}/'
            f'{len(HEATMAP_GRID_X) * len(HEATMAP_GRID_Y)} supported grid cells '
            '| white = insufficient local support'
            + (
                f' | dashed contour = {detection_threshold:.1f} working detection boundary'
                if detection_threshold is not None else ''
            )
        ),
        showarrow=False, font=dict(size=11, color='#53657a'),
    )
    figure.update_layout(
        title=title, width=900, height=700, template='plotly_white',
        margin=dict(l=75, r=135, t=90, b=95),
    )
    return figure

before_heatmap_figure = presentation_heatmap(
    weighted_before_grid,
    'Heatmap: before temporal logic',
    'Original mint<br>evidence (0-1)',
)
before_heatmap_figure.show()

after_heatmap_figure = presentation_heatmap(
    weighted_active_grid,
    'Heatmap: after temporal logic',
    'Presence-gated<br>active-mint<br>evidence (0-1)',
    detection_threshold=0.5,
)
# Superseded in the presentation by the agreement-weighted lag-consensus
# heatmap below. The single-lag grid remains available for the audit metrics.


Heatmap support: 534/899 grid cells; white means fewer than two retained readings within 3 cm.
Height audit: 60 below 1.0 cm (49 between 0.75 and 1.0 cm); 0 above 3.5 cm.
Balanced support: 787/899 grid cells (87.5%); strict support was 59.4%.


**ELI5.** The before/after charts now use the same visual grammar. Light blue means the current mint signal: the original classifier before temporal logic and **presence-gated active mint entering now** after temporal logic. Light green diamonds mean **mint decreasing / recovery**. Magenta is snout height on the right-hand axis. Keeping all three time series together makes their timing directly comparable; the dotted horizontal height limits remain 1.0 and 3.5 cm.

**Required deployment rule.** Active mint is allowed through only when the frozen odor-presence model scores at least 0.5. Below 0.5, displayed active-mint evidence is set to zero before any spatial gridding or smoothing. This is a hierarchical rule: first establish that an odor is present, then ask whether the temporal mint state is active. Recovery is not presence-gated because a real residual response can persist while current odor presence falls. On the known-clean first lane, odor-presence evidence had median 0.047 and 0/34 readings reached 0.5, so the gate removes all 34 startup false positives without changing either fitted model.

The three state scores compete with one another: active mint, mint recovery, and no active mint share the model's total evidence at each reading. Therefore, an inverse-looking active/recovery relationship is expected partly from the classifier design. It is useful for state interpretation, but it is not independent proof that the two physical processes are perfectly inverse.

The two presentation heatmaps are titled **Heatmap: before temporal logic** and **Heatmap: after temporal logic**. They use the same 2.0 s working pose correction, 1 cm grid, 3 cm neighborhood, softly height-weighted readings, support rule, fixed source overlay, and 0-1 color scale. The after-temporal map additionally applies the required odor-presence gate at each raw reading before smoothing. White cells remain missing support rather than zero evidence. These scores rank model evidence and are not calibrated probabilities.

**Why the strict map had so much white.** All 61 height-rejected readings in the reviewed raster were closer than 1.0 cm; none were above 3.5 cm, and 52 were only slightly below the cutoff at 0.75-1.0 cm. The blank regions therefore mostly reflect a hard quality threshold cutting through an otherwise dense path, not failed camera-smell synchronization.

**Balanced presentation view.** Both presentation heatmaps keep the same 3 cm spatial radius and two-reading minimum, but give readings from 0.5-1.0 cm a continuous reliability weight that rises from 0 to 1. Readings inside 1.0-3.5 cm retain full weight; the two readings below 0.5 cm remain excluded. This recovers supported coverage without pretending every height is equally reliable or spatially spreading evidence farther than 3 cm. The strict 1.0-3.5 cm calculation remains in the notebook as the primary QC audit. Soft weighting reduces the cutoff artifact, but it does not mathematically remove the physical effect of snout height on odor concentration.

In [ ]:
def spatial_metrics(name, score_column):
    source = raster_qc['source_region'].to_numpy(bool)
    score = raster_qc[score_column].to_numpy(float)
    return {
        'score': name,
        'source median': np.median(score[source]),
        'outside median': np.median(score[~source]),
        'source rows >= 0.5': np.mean(score[source] >= 0.5),
        'outside rows >= 0.5': np.mean(score[~source] >= 0.5),
        'source-ranking AUC': roc_auc_score(source, score),
        'score-time Spearman rho': raster_qc[['elapsed_s', score_column]].corr(method='spearman').iloc[0, 1],
    }

spatial_comparison = pd.DataFrame([
    spatial_metrics('Original mint score', 'original_mint_score'),
    spatial_metrics(
        'Presence-gated temporal active-mint score', 'active_mint_gated_score'
    ),
])
display(spatial_comparison.style.format({
    'source median': '{:.3f}',
    'outside median': '{:.3f}',
    'source rows >= 0.5': '{:.1%}',
    'outside rows >= 0.5': '{:.1%}',
    'source-ranking AUC': '{:.3f}',
    'score-time Spearman rho': '{:+.2f}',
}))

active_inside = raster_qc.loc[raster_qc['source_region'], 'active_mint_gated_score']
active_outside = raster_qc.loc[~raster_qc['source_region'], 'active_mint_gated_score']
recovery_inside = raster_qc.loc[raster_qc['source_region'], 'mint_recovery_score']
recovery_outside = raster_qc.loc[~raster_qc['source_region'], 'mint_recovery_score']


,score,source median,outside median,source rows >= 0.5,outside rows >= 0.5,source-ranking AUC,score-time Spearman rho
0,Original mint score,0.979,0.851,68.8%,60.5%,0.672,+0.53
1,Presence-gated temporal active-mint score,1.000,0.001,100.0%,22.2%,0.938,+0.28


**ELI5.** Source-ranking AUC asks: if we randomly pick one source-region reading and one outside reading, how often does the source reading score higher? The outside-above-0.5 column shows how much apparent mint remains away from the strip. The score-time correlation checks whether the map is mainly painting "later in the experiment" instead of "closer to the source." The source location is still approximate, so these remain pilot metrics.

In [ ]:
# Collapse upward and downward passes onto a common coordinate: negative is
# before crossing the strip; positive is after crossing it.
crossings = raster_qc.sort_values('elapsed_s').copy()
dt = np.gradient(crossings['elapsed_s'].to_numpy(float))
dy = np.gradient(crossings['y'].to_numpy(float))
crossings['vertical_speed'] = pd.Series(dy / np.maximum(dt, 1e-6)).rolling(
    5, center=True, min_periods=1,
).median().to_numpy()
crossings['directed_distance_cm'] = (
    (crossings['y'] - source_center_y) * np.sign(crossings['vertical_speed'])
)
crossing_rows = crossings[
    crossings['x'].between(source_x0, source_x1)
    & crossings['vertical_speed'].abs().ge(0.15)
    & crossings['directed_distance_cm'].between(-12.0, 12.0)
].copy()
distance_edges = np.arange(-12.0, 12.01, 2.0)
crossing_rows['distance_bin'] = pd.cut(
    crossing_rows['directed_distance_cm'], distance_edges, include_lowest=True,
)
crossing_profile = crossing_rows.groupby('distance_bin', observed=True).agg(
    directed_distance_cm=('directed_distance_cm', 'median'),
    active_mint_gated_score=('active_mint_gated_score', 'median'),
    mint_recovery_score=('mint_recovery_score', 'median'),
    rows=('active_mint_gated_score', 'size'),
).reset_index()

crossing_figure = go.Figure()
crossing_figure.add_trace(go.Scatter(
    x=crossing_profile['directed_distance_cm'], y=crossing_profile['active_mint_gated_score'],
    mode='lines+markers', name='Presence-gated active mint',
    customdata=crossing_profile['rows'],
    hovertemplate='distance %{x:.1f} cm<br>active score %{y:.3f}<br>rows %{customdata}<extra></extra>',
))
crossing_figure.add_trace(go.Scatter(
    x=crossing_profile['directed_distance_cm'], y=crossing_profile['mint_recovery_score'],
    mode='lines+markers', name='Mint recovery', line=dict(dash='dash'),
    customdata=crossing_profile['rows'],
    hovertemplate='distance %{x:.1f} cm<br>recovery score %{y:.3f}<br>rows %{customdata}<extra></extra>',
))
crossing_figure.add_vrect(
    x0=-SOURCE_WIDTH_CM / 2, x1=SOURCE_WIDTH_CM / 2,
    fillcolor='rgba(63, 191, 143, 0.20)', line_width=0,
    annotation_text='reported strip width', annotation_position='top left',
)
crossing_figure.add_hline(y=0.5, line_dash='dot', line_width=1)
crossing_figure.update_xaxes(title='Directed distance from strip center (cm) | before < 0 < after')
crossing_figure.update_yaxes(title='Median dynamic score', range=[-0.03, 1.03])
crossing_figure.update_layout(
    title='Direction-normalized line crossings',
    height=500, width=1050, margin=dict(l=75, r=35, t=80, b=70),
)
crossing_figure.show()


**ELI5.** Every vertical lane is rotated conceptually so movement goes from left to right: negative distance is before reaching the horizontal strip, zero is the strip center, and positive distance is after leaving it. If active evidence peaks near zero while recovery evidence becomes stronger afterward, the temporal model is behaving in the intended direction. This diagnostic uses only moving, height-qualified readings over the reported 17 cm source length.

## Lag-sensitivity audit against the reported strip

This audit remaps the **same frozen model scores and same 157 reviewed Cyranose readings** to camera poses from 0 to 5 seconds earlier in 0.25-second steps. The reported 17 x 3 cm strip footprint is held fixed; it is not recomputed for each lag.

The primary criterion is the active-score-weighted distance outside the reported strip, calculated from raw mapped readings before height rejection, gridding, or smoothing. Lower is closer to the physical target. Source-ranking AUC and the fraction of outside readings above 0.5 are secondary checks. The lag is therefore selected numerically rather than from whichever heatmap looks nicest.

This is a **calibration analysis**, not independent validation: the strip location is being used to estimate lag, and its desk coordinates are approximate. A lag chosen here must be frozen and tested unchanged on a new spatial run.

In [ ]:
LAG_AUDIT_VALUES_S = np.arange(0.0, 5.01, 0.25)

def align_scored_readings_at_lag(lag_s):
    rows = []
    for reading_index, reading in scored.iterrows():
        if pd.isna(reading.get('pose_elapsed_s')):
            continue
        corrected_time = float(reading['pose_elapsed_s']) - float(lag_s)
        pose_index = int(np.searchsorted(pose_times, corrected_time))
        if pose_index == 0 or pose_index == len(pose_times):
            continue
        gap = pose_times[pose_index] - pose_times[pose_index - 1]
        if gap <= 0 or gap > MAX_INTERPOLATION_GAP_S:
            continue
        fraction = (corrected_time - pose_times[pose_index - 1]) / gap
        x, y, z = pose_xyz[pose_index - 1] + fraction * (
            pose_xyz[pose_index] - pose_xyz[pose_index - 1]
        )
        elapsed_s = float(reading['pose_elapsed_s'])
        if not (xmin <= x <= xmax and ymin <= y <= ymax):
            continue
        if not (RASTER_WINDOW_S[0] <= elapsed_s <= RASTER_WINDOW_S[1]):
            continue
        presence_pass = (
            float(reading['odor_presence_score']) >= ODOR_PRESENCE_GATE_THRESHOLD
        )
        active_gated = float(reading['active_mint_score']) if presence_pass else 0.0
        rows.append({
            'reading_index': int(reading_index),
            'elapsed_s': elapsed_s,
            'x': float(x), 'y': float(y), 'z': float(z),
            'height': abs(float(z)),
            'odor_presence_score': float(reading['odor_presence_score']),
            'active_mint_score': float(reading['active_mint_score']),
            'active_mint_gated_score': active_gated,
            'mint_recovery_score': float(reading['mint_recovery_score']),
        })
    return pd.DataFrame(rows).sort_values('elapsed_s').reset_index(drop=True)

lag_audit_frames = {
    float(lag_s): align_scored_readings_at_lag(float(lag_s))
    for lag_s in LAG_AUDIT_VALUES_S
}
common_reading_indices = set.intersection(*(
    set(frame['reading_index']) for frame in lag_audit_frames.values()
))
assert len(common_reading_indices) == len(raster_window) == 157

def distance_outside_reported_strip(y_values):
    y_values = np.asarray(y_values, dtype=float)
    return np.maximum.reduce([
        source_y0 - y_values,
        y_values - source_y1,
        np.zeros_like(y_values),
    ])

def weighted_strip_distance(frame, mask=None):
    if mask is None:
        mask = np.ones(len(frame), dtype=bool)
    mask = np.asarray(mask, dtype=bool) & frame['x'].between(source_x0, source_x1).to_numpy()
    weights = frame.loc[mask, 'active_mint_gated_score'].to_numpy(float)
    if weights.sum() <= 0:
        return np.nan
    distances = distance_outside_reported_strip(frame.loc[mask, 'y'])
    return float(np.average(distances, weights=weights))

lag_audit_rows = []
for lag_s, frame in lag_audit_frames.items():
    frame = frame[frame['reading_index'].isin(common_reading_indices)].copy()
    source_mask = (
        frame['x'].between(source_x0, source_x1)
        & frame['y'].between(source_y0, source_y1)
    )
    score = frame['active_mint_gated_score'].to_numpy(float)
    elapsed = frame['elapsed_s'].to_numpy(float)
    vertical_speed = np.gradient(frame['y'].to_numpy(float)) / np.maximum(
        np.gradient(elapsed), 1e-6
    )
    vertical_speed = pd.Series(vertical_speed).rolling(
        5, center=True, min_periods=1
    ).median().to_numpy()
    moving = np.abs(vertical_speed) >= 0.15
    lag_audit_rows.append({
        'lag_s': lag_s,
        'active-weighted distance outside strip (cm)': weighted_strip_distance(frame),
        'upward-pass distance (cm)': weighted_strip_distance(
            frame, moving & (vertical_speed < 0)
        ),
        'downward-pass distance (cm)': weighted_strip_distance(
            frame, moving & (vertical_speed > 0)
        ),
        'source-ranking AUC': roc_auc_score(source_mask, score),
        'outside rows >= 0.5': float(np.mean(score[~source_mask] >= 0.5)),
        'source rows >= 0.5': float(np.mean(score[source_mask] >= 0.5)),
        'source rows': int(source_mask.sum()),
    })

lag_audit = pd.DataFrame(lag_audit_rows)
distance_column = 'active-weighted distance outside strip (cm)'
best_distance = float(lag_audit[distance_column].min())
best_auc = float(lag_audit['source-ranking AUC'].max())
supported_lag_rows = lag_audit[
    lag_audit[distance_column].le(best_distance + 0.02)
    & lag_audit['source-ranking AUC'].ge(best_auc - 0.005)
].copy()
PROVISIONAL_LAG_MIN_S = float(supported_lag_rows['lag_s'].min())
PROVISIONAL_LAG_MAX_S = float(supported_lag_rows['lag_s'].max())
PRIMARY_LAG_MINIMUM_S = float(
    lag_audit.loc[lag_audit[distance_column].idxmin(), 'lag_s']
)

audit_figure = make_subplots(
    rows=1, cols=3,
    subplot_titles=(
        'Raw active evidence distance from strip (lower is better)',
        'Source-versus-outside ranking (higher is better)',
        'Outside readings above 0.5 (lower is better)',
    ),
    horizontal_spacing=0.08,
)
for metric, name, color, dash in [
    (distance_column, 'All passes', '#237a57', 'solid'),
    ('upward-pass distance (cm)', 'Upward passes', '#3b82a0', 'dot'),
    ('downward-pass distance (cm)', 'Downward passes', '#c06a3b', 'dash'),
]:
    audit_figure.add_trace(go.Scatter(
        x=lag_audit['lag_s'], y=lag_audit[metric],
        mode='lines+markers', name=name,
        line=dict(color=color, width=2, dash=dash), marker=dict(size=6),
        hovertemplate='lag %{x:.2f} s<br>distance %{y:.3f} cm<extra></extra>',
    ), row=1, col=1)
audit_figure.add_trace(go.Scatter(
    x=lag_audit['lag_s'], y=lag_audit['source-ranking AUC'],
    mode='lines+markers', name='Source-ranking AUC',
    line=dict(color='#6b5ca5', width=2), marker=dict(size=6),
    hovertemplate='lag %{x:.2f} s<br>AUC %{y:.3f}<extra></extra>',
), row=1, col=2)
audit_figure.add_trace(go.Scatter(
    x=lag_audit['lag_s'], y=lag_audit['outside rows >= 0.5'],
    mode='lines+markers', name='Outside rows >= 0.5',
    line=dict(color='#9c4f64', width=2), marker=dict(size=6),
    hovertemplate='lag %{x:.2f} s<br>outside rows %{y:.1%}<extra></extra>',
), row=1, col=3)
for plot_col in range(1, 4):
    audit_figure.add_vrect(
        x0=PROVISIONAL_LAG_MIN_S, x1=PROVISIONAL_LAG_MAX_S,
        fillcolor='rgba(35, 122, 87, 0.12)', line_width=0,
        row=1, col=plot_col,
    )
    audit_figure.add_vline(
        x=3.0, line_dash='dash', line_color='#4b5563', line_width=1.5,
        row=1, col=plot_col,
    )
audit_figure.update_xaxes(title_text='Assumed physical lag (s)')
audit_figure.update_yaxes(title_text='Distance (cm)', row=1, col=1)
audit_figure.update_yaxes(title_text='AUC', range=[0.45, 1.01], row=1, col=2)
audit_figure.update_yaxes(title_text='Fraction', tickformat='.0%', row=1, col=3)
audit_figure.update_layout(
    title=(
        'Lag audit against fixed reported strip | shaded = numerically supported band | '
        'dashed line = previous 3.0 s'
    ),
    width=1450, height=520, template='plotly_white',
    margin=dict(l=70, r=40, t=100, b=70),
    legend=dict(orientation='h', yanchor='bottom', y=1.03, xanchor='left', x=0),
)
audit_figure.show()

audit_table_lags = [0.0, 1.0, 1.5, 2.0, 2.25, 2.5, 2.75, 3.0, 4.0, 5.0]
display(lag_audit[lag_audit['lag_s'].isin(audit_table_lags)].style.format({
    'lag_s': '{:.2f}',
    distance_column: '{:.3f}',
    'upward-pass distance (cm)': '{:.3f}',
    'downward-pass distance (cm)': '{:.3f}',
    'source-ranking AUC': '{:.3f}',
    'outside rows >= 0.5': '{:.1%}',
    'source rows >= 0.5': '{:.1%}',
}))


,lag_s,active-weighted distance outside strip (cm),upward-pass distance (cm),downward-pass distance (cm),source-ranking AUC,outside rows >= 0.5,source rows >= 0.5,source rows
0,0.00,3.665,3.732,3.353,0.679,23.2%,73.3%,15
4,1.00,2.220,2.208,2.221,0.873,21.1%,93.3%,15
6,1.50,1.764,1.618,1.853,0.951,20.4%,100.0%,15
8,2.00,1.504,1.261,1.678,0.958,19.9%,100.0%,16
9,2.25,1.448,1.155,1.652,0.967,20.4%,100.0%,15
10,2.50,1.451,1.151,1.662,0.967,18.1%,100.0%,19
11,2.75,1.496,1.160,1.728,0.953,20.4%,100.0%,15
12,3.00,1.595,1.288,1.818,0.945,18.7%,100.0%,18
16,4.00,2.487,2.578,2.402,0.860,22.1%,76.5%,17
20,5.00,3.435,4.150,2.704,0.758,26.8%,40.0%,15


In [ ]:
LAG_MAP_VALUES_S = (0.0, 1.5, PRIMARY_LAG_MINIMUM_S, 3.0)
lag_map_figure = make_subplots(
    rows=2, cols=2,
    subplot_titles=tuple(f'Lag {lag_s:.2f} s' for lag_s in LAG_MAP_VALUES_S),
    horizontal_spacing=0.08, vertical_spacing=0.10,
)
lag_map_support = {}
for panel_index, lag_s in enumerate(LAG_MAP_VALUES_S):
    row = panel_index // 2 + 1
    col = panel_index % 2 + 1
    frame = lag_audit_frames[float(lag_s)].copy()
    grid, support = smooth_height_weighted_grid(
        frame, 'active_mint_gated_score'
    )
    lag_map_support[lag_s] = sum(
        value is not None for grid_row in grid for value in grid_row
    )
    lag_map_figure.add_trace(go.Heatmap(
        x=HEATMAP_GRID_X, y=HEATMAP_GRID_Y, z=grid,
        coloraxis='coloraxis', connectgaps=False,
        hovertemplate=(
            'Desk X %{x:.1f} cm<br>Desk Y %{y:.1f} cm'
            f'<br>Lag {lag_s:.2f} s active evidence %{{z:.3f}}<extra></extra>'
        ),
    ), row=row, col=col)
    lag_map_figure.add_trace(go.Contour(
        x=HEATMAP_GRID_X, y=HEATMAP_GRID_Y, z=grid,
        contours=dict(start=0.5, end=0.5, size=1, coloring='none'),
        line=dict(color='#394150', width=1.5, dash='dash'),
        showscale=False, hoverinfo='skip',
    ), row=row, col=col)
    add_source_overlay(lag_map_figure, row=row, col=col)
    lag_map_figure.update_xaxes(
        title_text='Desk X (cm)', autorange='reversed', row=row, col=col,
    )
    lag_map_figure.update_yaxes(
        title_text='Desk Y (cm)' if col == 1 else None,
        autorange='reversed', row=row, col=col,
    )
lag_map_figure.update_layout(
    title=(
        'Lag sensitivity heatmaps | identical model scores, fixed strip, '
        'soft height weighting, and 3 cm smoothing'
    ),
    width=1150, height=1000, template='plotly_white', showlegend=False,
    coloraxis=dict(
        colorscale='Cividis', cmin=0, cmax=1,
        colorbar=dict(title='Presence-gated<br>active mint<br>(0-1)', x=1.02),
    ),
    margin=dict(l=70, r=150, t=110, b=70),
)
lag_map_figure.show()

primary_row = lag_audit.loc[
    lag_audit['lag_s'].eq(PRIMARY_LAG_MINIMUM_S)
].iloc[0]
old_row = lag_audit.loc[lag_audit['lag_s'].eq(3.0)].iloc[0]
best_upward_lag = float(lag_audit.loc[
    lag_audit['upward-pass distance (cm)'].idxmin(), 'lag_s'
])
best_downward_lag = float(lag_audit.loc[
    lag_audit['downward-pass distance (cm)'].idxmin(), 'lag_s'
])
display(Markdown(f'''\
**Audit result.** The raw-distance minimum is **{PRIMARY_LAG_MINIMUM_S:.2f} s**. 
The distance and AUC criteria jointly support a narrow **{PROVISIONAL_LAG_MIN_S:.2f}-{PROVISIONAL_LAG_MAX_S:.2f} s** band rather than the previous 3.00 s assumption. 
At the primary minimum, active-score-weighted distance outside the strip is **{primary_row[distance_column]:.3f} cm** and source-ranking AUC is **{primary_row['source-ranking AUC']:.3f}**; at 3.00 s they are **{old_row[distance_column]:.3f} cm** and **{old_row['source-ranking AUC']:.3f}**. 
The separate pass-direction minima are **{best_upward_lag:.2f} s upward** and **{best_downward_lag:.2f} s downward**.

**Interpretation.** This evidence makes 3.00 s look somewhat overcorrected for this raster, but it does not establish a universal exact lag. The strip coordinates are approximate, and any difference between upward and downward optima indicates that one fixed delay cannot perfectly explain sensor persistence, changing speed, plume transport, and manual turns. Treat **{PROVISIONAL_LAG_MIN_S:.2f}-{PROVISIONAL_LAG_MAX_S:.2f} s** as a provisional calibration range, then test it unchanged on a new run with independently measured strip coordinates.
'''))


**Audit result.** The raw-distance minimum is **2.25 s**. 
The distance and AUC criteria jointly support a narrow **2.25-2.50 s** band rather than the previous 3.00 s assumption. 
At the primary minimum, active-score-weighted distance outside the strip is **1.448 cm** and source-ranking AUC is **0.967**; at 3.00 s they are **1.595 cm** and **0.945**. 
The separate pass-direction minima are **2.50 s upward** and **2.25 s downward**.

**Interpretation.** This evidence makes 3.00 s look somewhat overcorrected for this raster, but it does not establish a universal exact lag. The strip coordinates are approximate, and any difference between upward and downward optima indicates that one fixed delay cannot perfectly explain sensor persistence, changing speed, plume transport, and manual turns. Treat **2.25-2.50 s** as a provisional calibration range, then test it unchanged on a new run with independently measured strip coordinates.


## Ceiling check: lag-consensus map

A single fixed lag can move evidence to one attractive location by chance. This final refinement therefore combines the already-declared **1.50, 1.75, 2.00, and 2.25 s** working range without choosing the prettiest result. Every lag uses the same frozen scores, pose data, height weighting, 3 cm spatial kernel, and support rule. The consensus is the cell-wise median where at least three of four lag maps have support. No rectangle mask, source clipping, or geometry prior is used.

In [ ]:
CONSENSUS_LAGS_S = (1.50, 1.75, 2.00, 2.25)
MIN_CONSENSUS_LAG_SUPPORT = 3

def grid_to_float_array(grid):
    return np.array([
        [np.nan if value is None else float(value) for value in row]
        for row in grid
    ], dtype=float)

consensus_lag_grids = {}
for lag_s in CONSENSUS_LAGS_S:
    lag_grid, _ = smooth_height_weighted_grid(
        lag_audit_frames[float(lag_s)], 'active_mint_gated_score'
    )
    consensus_lag_grids[float(lag_s)] = grid_to_float_array(lag_grid)

lag_grid_stack = np.stack([
    consensus_lag_grids[float(lag_s)] for lag_s in CONSENSUS_LAGS_S
])
lag_valid_count = np.isfinite(lag_grid_stack).sum(axis=0)
lag_consensus_grid = np.ma.median(
    np.ma.masked_invalid(lag_grid_stack), axis=0
).filled(np.nan)
lag_consensus_grid[lag_valid_count < MIN_CONSENSUS_LAG_SUPPORT] = np.nan
lag_detection_agreement = np.divide(
    np.sum((lag_grid_stack >= 0.5) & np.isfinite(lag_grid_stack), axis=0),
    lag_valid_count,
    out=np.full(lag_valid_count.shape, np.nan, dtype=float),
    where=lag_valid_count > 0,
)
lag_detection_agreement[lag_valid_count < MIN_CONSENSUS_LAG_SUPPORT] = np.nan
agreement_weighted_consensus = lag_consensus_grid * lag_detection_agreement
working_grid = consensus_lag_grids[float(RESPONSE_LAG_S)]

consensus_supported_cells = int(np.isfinite(agreement_weighted_consensus).sum())
consensus_figure = make_subplots(rows=1, cols=1)
consensus_figure.add_trace(go.Heatmap(
    x=HEATMAP_GRID_X, y=HEATMAP_GRID_Y, z=agreement_weighted_consensus,
    colorscale='Cividis', zmin=0, zmax=1, connectgaps=False,
    colorbar=dict(title='<br>active-mint<br>evidence (0-1)', len=0.82),
    hovertemplate=(
        'Desk X %{x:.1f} cm<br>Desk Y %{y:.1f} cm'
        '<br>Robust active-mint evidence %{z:.3f}<extra></extra>'
    ),
), row=1, col=1)
consensus_figure.add_trace(go.Contour(
    x=HEATMAP_GRID_X, y=HEATMAP_GRID_Y, z=agreement_weighted_consensus,
    contours=dict(start=0.5, end=0.5, size=1, coloring='none'),
    line=dict(color='#394150', width=2, dash='dash'),
    showscale=False, hoverinfo='skip',
), row=1, col=1)
add_source_overlay(consensus_figure, row=1, col=1)
consensus_figure.update_xaxes(
    title_text='Desk X (cm)', autorange='reversed', row=1, col=1,
)
consensus_figure.update_yaxes(
    title_text='Desk Y (cm)', autorange='reversed',
    scaleanchor='x', scaleratio=1, row=1, col=1,
)
consensus_figure.add_annotation(
    x=0, y=-0.13, xref='paper', yref='paper', xanchor='left',
    text=(
        f'Median evidence x cross-lag agreement | {consensus_supported_cells}/'
        f'{len(HEATMAP_GRID_X) * len(HEATMAP_GRID_Y)} supported grid cells | '
        'at least 3 of 4 lags required | no source-shape prior'
    ),
    showarrow=False, font=dict(size=11, color='#53657a'),
)
consensus_figure.update_layout(
    title='Heatmap: after temporal logic',
    width=900, height=700, template='plotly_white', showlegend=False,
    margin=dict(l=75, r=150, t=90, b=95),
)
consensus_figure.show()

grid_x_mesh, grid_y_mesh = np.meshgrid(HEATMAP_GRID_X, HEATMAP_GRID_Y)
reported_source_grid = (
    (grid_x_mesh >= source_x0) & (grid_x_mesh <= source_x1)
    & (grid_y_mesh >= source_y0) & (grid_y_mesh <= source_y1)
)

def grid_ceiling_metrics(grid):
    valid = np.isfinite(grid)
    detected = valid & (grid >= 0.5)
    detected_count = int(detected.sum())
    outside_fraction = (
        float((detected & ~reported_source_grid).sum() / detected_count)
        if detected_count else np.nan
    )
    within_source_x = detected & (grid_x_mesh >= source_x0) & (grid_x_mesh <= source_x1)
    top_extension = (
        max(0.0, float(source_y0 - grid_y_mesh[within_source_x].min()))
        if within_source_x.any() else np.nan
    )
    return {
        'supported cells': int(valid.sum()),
        'cells >= 0.5': detected_count,
        'detected cells outside reported strip': outside_fraction,
        'extension above strip within source X (cm)': top_extension,
    }

ceiling_metrics = pd.DataFrame({
    f'Working {RESPONSE_LAG_S:.2f} s': grid_ceiling_metrics(working_grid),
    'Lag-consensus': grid_ceiling_metrics(lag_consensus_grid),
    'Agreement-weighted consensus': grid_ceiling_metrics(agreement_weighted_consensus),
}).T
display(ceiling_metrics.style.format({
    'supported cells': '{:.0f}',
    'cells >= 0.5': '{:.0f}',
    'detected cells outside reported strip': '{:.1%}',
    'extension above strip within source X (cm)': '{:.1f}',
}))


,supported cells,cells >= 0.5,detected cells outside reported strip,extension above strip within source X (cm)
Working 2.00 s,787,171,70.8%,7.3
Lag-consensus,771,171,70.2%,7.3
Agreement-weighted consensus,771,144,66.7%,6.3


In [ ]:
working_ceiling = grid_ceiling_metrics(working_grid)
consensus_ceiling = grid_ceiling_metrics(lag_consensus_grid)
robust_ceiling = grid_ceiling_metrics(agreement_weighted_consensus)
display(Markdown(f'''\
**ELI5.** Imagine making the same smell map with four reasonable timing corrections. At each supported location, we first take the typical answer (the median). We then ask how many of the four maps agree that active mint is at least 0.5. The displayed value multiplies the typical answer by that agreement. A location therefore stays bright only when the mint evidence is both strong **and** repeatable across the timing choices; a feature caused mainly by one exact lag becomes dimmer.

**Observed ceiling check.** The consensus retains **{consensus_ceiling['supported cells']:.0f} supported cells**. Its above-strip extension within the strip's horizontal span is **{consensus_ceiling['extension above strip within source X (cm)']:.1f} cm**, versus **{working_ceiling['extension above strip within source X (cm)']:.1f} cm** for the 2.00 s map. The fraction of thresholded cells outside the approximate reported strip is **{consensus_ceiling['detected cells outside reported strip']:.1%}** versus **{working_ceiling['detected cells outside reported strip']:.1%}**.

The agreement-weighted ceiling view reduces the above-strip extension to **{robust_ceiling['extension above strip within source X (cm)']:.1f} cm** and the outside-strip thresholded fraction to **{robust_ceiling['detected cells outside reported strip']:.1%}**. This is a **robustness visualization, not a new trained model**. It does not use the reported rectangle to reshape or suppress the field. If this view remains irregular, that residual morphology is the honest ceiling of what timing uncertainty alone can repair in this run.
'''))


**ELI5.** Imagine making the same smell map with four reasonable timing corrections. At each supported location, we first take the typical answer (the median). We then ask how many of the four maps agree that active mint is at least 0.5. The displayed value multiplies the typical answer by that agreement. A location therefore stays bright only when the mint evidence is both strong **and** repeatable across the timing choices; a feature caused mainly by one exact lag becomes dimmer.

**Observed ceiling check.** The consensus retains **771 supported cells**. Its above-strip extension within the strip's horizontal span is **7.3 cm**, versus **7.3 cm** for the 2.00 s map. The fraction of thresholded cells outside the approximate reported strip is **70.2%** versus **70.8%**.

The agreement-weighted ceiling view reduces the above-strip extension to **6.3 cm** and the outside-strip thresholded fraction to **66.7%**. This is a **robustness visualization, not a new trained model**. It does not use the reported rectangle to reshape or suppress the field. If this view remains irregular, that residual morphology is the honest ceiling of what timing uncertainty alone can repair in this run.


## Freeze temporal mint-seeker v1

Freezing means the fitted models, sensor order, causal history configuration, regularization, baseline envelope, and training-data digest are saved under one version. Future shape experiments must load this artifact unchanged. Horizontal line raster-454 is used only for the reload test below; it is not added to the training data or used to alter the model.

In [ ]:
FROZEN_MODEL_VERSION = 'temporal-mint-seeker-v1'
frozen_model_dir = (
    ROOT / 'experiments' / 'classifier' / 'mint-identity-pilot-01'
    / 'frozen-models' / FROZEN_MODEL_VERSION
)
frozen_model_dir.mkdir(parents=True, exist_ok=True)
frozen_model_path = frozen_model_dir / 'model.joblib'
frozen_card_path = frozen_model_dir / 'model_card.json'
frozen_checksum_path = frozen_model_dir / 'model.sha256'
frozen_policy_path = frozen_model_dir / 'deployment_policy.json'

training_paths = []
for odor_class, pattern in TRIAL_PATTERNS.items():
    training_paths.extend(sorted((RAW / odor_class).glob(pattern)))
assert len(training_paths) == 70

training_hasher = hashlib.sha256()
for path in training_paths:
    training_hasher.update(path.relative_to(ROOT).as_posix().encode('utf-8'))
    training_hasher.update(b'\0')
    training_hasher.update(path.read_bytes())
training_dataset_sha256 = training_hasher.hexdigest()
training_manifest_path = ROOT / 'experiments' / 'classifier' / 'mint-identity-pilot-01' / 'trial_manifest.csv'
training_manifest_sha256 = hashlib.sha256(training_manifest_path.read_bytes()).hexdigest()
training_trial_counts = {
    key: int(value)
    for key, value in metadata.groupby('odor_class')['trial'].nunique().to_dict().items()
}

frozen_bundle = {
    'artifact_version': FROZEN_MODEL_VERSION,
    'frozen_on': '2026-07-23',
    'temporal_model': temporal_model,
    'odor_presence_model': presence_model,
    'mint_identity_model': identity_model,
    'selected_feature_name': selected_feature,
    'feature_config': FEATURE_CONFIGS[selected_feature],
    'regularization_c': selected_c,
    'minimum_history_s': MIN_HISTORY_S,
    'state_order': STATE_ORDER.tolist(),
    'stationary_sensor_columns': SENSORS,
    'direct_serial_sensor_columns': [f'pcnose_S{i}_kohm' for i in range(1, 33)],
    'baseline_rms_limit_pct': baseline_rms_limit,
    'training_trial_counts': training_trial_counts,
    'training_dataset_sha256': training_dataset_sha256,
    'training_manifest_sha256': training_manifest_sha256,
    'response_definition': '100 * (resistance - flag1_baseline) / flag1_baseline',
    'deployment_rule': 'Load unchanged for new spatial shapes; create a new version to retrain.',
}

if not frozen_model_path.exists():
    joblib.dump(frozen_bundle, frozen_model_path, compress=3)

loaded_frozen_bundle = joblib.load(frozen_model_path)
assert loaded_frozen_bundle['artifact_version'] == FROZEN_MODEL_VERSION
assert loaded_frozen_bundle['training_dataset_sha256'] == training_dataset_sha256
assert loaded_frozen_bundle['training_manifest_sha256'] == training_manifest_sha256
assert loaded_frozen_bundle['selected_feature_name'] == selected_feature
assert loaded_frozen_bundle['feature_config'] == FEATURE_CONFIGS[selected_feature]
assert np.isclose(loaded_frozen_bundle['regularization_c'], selected_c)
deployment_policy = json.loads(frozen_policy_path.read_text(encoding='utf-8'))
assert deployment_policy['model_artifact_version'] == FROZEN_MODEL_VERSION
assert np.isclose(
    deployment_policy['odor_presence_threshold'], ODOR_PRESENCE_GATE_THRESHOLD
)
assert np.isclose(
    deployment_policy['working_spatial_pose_lag_s'], RESPONSE_LAG_S
)

# Reload test: the frozen artifact must reproduce all three temporal raster
# scores and both current-state classifier stages before it is accepted.
reloaded_temporal_scores = aligned_proba(
    loaded_frozen_bundle['temporal_model'], dynamic_x
)
assert np.allclose(reloaded_temporal_scores, dynamic_scores, rtol=1e-12, atol=1e-12)
assert np.allclose(
    loaded_frozen_bundle['odor_presence_model'].predict_proba(original_scan_features),
    presence_model.predict_proba(original_scan_features),
    rtol=1e-12, atol=1e-12,
)
assert np.allclose(
    loaded_frozen_bundle['mint_identity_model'].predict_proba(original_direction),
    identity_model.predict_proba(original_direction),
    rtol=1e-12, atol=1e-12,
)

frozen_model_sha256 = hashlib.sha256(frozen_model_path.read_bytes()).hexdigest()
if frozen_checksum_path.exists():
    assert frozen_checksum_path.read_text(encoding='utf-8').strip() == frozen_model_sha256
else:
    frozen_checksum_path.write_text(f'{frozen_model_sha256}\n', encoding='utf-8')

frozen_validation_row = validation_summary.loc[
    validation_summary['model'].eq('Temporal history')
].iloc[0]
frozen_spatial_row = spatial_comparison.loc[
    spatial_comparison['score'].eq('Presence-gated temporal active-mint score')
].iloc[0]
model_card = {
    'artifact_version': FROZEN_MODEL_VERSION,
    'status': 'frozen pilot; independent spatial validation required',
    'frozen_on': '2026-07-23',
    'source_notebook': 'analysis/notebooks/06_dynamic_mint_exposure_recovery.ipynb',
    'artifact_file': 'model.joblib',
    'artifact_sha256': frozen_model_sha256,
    'deployment_policy_file': 'deployment_policy.json',
    'odor_presence_gate_threshold': ODOR_PRESENCE_GATE_THRESHOLD,
    'working_spatial_pose_lag_s': RESPONSE_LAG_S,
    'working_spatial_pose_lag_rationale': (
        'Conservative tested compromise between 1.5 and 2.25 s after the '
        'fixed-ground-truth lag audit; not a universal physical constant.'
    ),
    'training_dataset_sha256': training_dataset_sha256,
    'training_manifest_sha256': training_manifest_sha256,
    'training_trial_counts': training_trial_counts,
    'total_independent_trials': int(sum(training_trial_counts.values())),
    'state_order': STATE_ORDER.tolist(),
    'selected_feature_name': selected_feature,
    'feature_config': {
        key: list(value) for key, value in FEATURE_CONFIGS[selected_feature].items()
    },
    'regularization_c': selected_c,
    'minimum_history_s': MIN_HISTORY_S,
    'baseline_rms_limit_pct': baseline_rms_limit,
    'grouped_validation_balanced_accuracy': float(frozen_validation_row['balanced accuracy']),
    'grouped_validation_active_mint_auc': float(frozen_validation_row['active-mint AUC']),
    'grouped_validation_recovery_auc': float(frozen_validation_row['recovery AUC']),
    'line_raster_454_presence_gated_source_ranking_auc': float(
        frozen_spatial_row['source-ranking AUC']
    ),
    'line_raster_454_presence_gated_outside_rows_above_0_5': float(
        frozen_spatial_row['outside rows >= 0.5']
    ),
    'line_raster_454_presence_gated_score_time_spearman_rho': float(
        frozen_spatial_row['score-time Spearman rho']
    ),
    'line_raster_454_known_clean_startup_rows': int(
        deployment_policy['known_clean_first_lane_audit']['readings']
    ),
    'line_raster_454_known_clean_startup_rows_passing_gate': int(
        deployment_policy['known_clean_first_lane_audit']['readings_passing_presence_gate']
    ),
    'line_raster_454_role': 'reload test and pilot evaluation only; not training or tuning',
    'python_version': platform.python_version(),
    'numpy_version': np.__version__,
    'pandas_version': pd.__version__,
    'scikit_learn_version': sklearn.__version__,
    'limitations': [
        'Mint identity is established only against blank, bergamot, and lemongrass in this pilot.',
        'Only 15 independent mint exposure/recovery trials are available.',
        'Stationary PCnose+ phases differ from open-air moving raster scans.',
        'Model scores are not independently calibrated probabilities.',
    ],
    'next_test': 'Apply unchanged to new mint shapes and randomized-order crossings.',
}
if frozen_card_path.exists():
    existing_card = json.loads(frozen_card_path.read_text(encoding='utf-8'))
    assert existing_card['artifact_sha256'] == frozen_model_sha256
    assert existing_card['training_dataset_sha256'] == training_dataset_sha256
    assert existing_card['deployment_policy_file'] == 'deployment_policy.json'
    assert np.isclose(
        existing_card['odor_presence_gate_threshold'], ODOR_PRESENCE_GATE_THRESHOLD
    )
frozen_card_path.write_text(
    json.dumps(model_card, indent=2) + '\n', encoding='utf-8'
)

print('Frozen model:', frozen_model_path.relative_to(ROOT))
print('Model SHA-256:', frozen_model_sha256)
print('Deployment policy:', frozen_policy_path.relative_to(ROOT))
print('Training trials:', training_trial_counts)
print('Reload verification: exact prediction match passed')


Frozen model: experiments\classifier\mint-identity-pilot-01\frozen-models\temporal-mint-seeker-v1\model.joblib
Model SHA-256: 7a3c3a6e078073f9e0c7e92ba7c3ec91ac0a04f64a12e6bb2f646f60cc4cb2b2
Deployment policy: experiments\classifier\mint-identity-pilot-01\frozen-models\temporal-mint-seeker-v1\deployment_policy.json
Training trials: {'bergamot': 20, 'blank': 15, 'lemongrass': 20, 'mint': 15}
Reload verification: exact prediction match passed


In [ ]:
instant_row = validation_summary.loc[validation_summary['model'].eq('Current-only snapshot')].iloc[0]
temporal_row = validation_summary.loc[validation_summary['model'].eq('Temporal history')].iloc[0]
original_spatial = spatial_comparison.loc[spatial_comparison['score'].eq('Original mint score')].iloc[0]
temporal_spatial = spatial_comparison.loc[
    spatial_comparison['score'].eq('Presence-gated temporal active-mint score')
].iloc[0]

display(Markdown(f"""
## Interpretation

Complete-trial nested validation improves balanced state accuracy from **{instant_row['balanced accuracy']:.1%}** for a current-only snapshot to **{temporal_row['balanced accuracy']:.1%}** with causal temporal history. The locked deployment configuration is **{selected_feature}** with regularization `C={selected_c:g}`. This choice was made without consulting the raster.

On **{RASTER_DISPLAY_NAME}**, the original classifier's approximate source-versus-outside medians are **{original_spatial['source median']:.3f}** versus **{original_spatial['outside median']:.3f}**. After temporal logic plus the frozen odor-presence gate, active-mint medians are **{temporal_spatial['source median']:.3f}** versus **{temporal_spatial['outside median']:.3f}**. The fraction of outside readings above 0.5 changes from **{original_spatial['outside rows >= 0.5']:.1%}** to **{temporal_spatial['outside rows >= 0.5']:.1%}**, while source-ranking AUC changes from **{original_spatial['source-ranking AUC']:.3f}** to **{temporal_spatial['source-ranking AUC']:.3f}**. Score-time correlation changes from **{original_spatial['score-time Spearman rho']:+.2f}** to **{temporal_spatial['score-time Spearman rho']:+.2f}**.

**Decision:** the temporal model is useful only if it improves complete-trial state separation and produces a more source-aligned active score without being selected against this map. The current experiment remains a pilot because there are only 15 independent mint recovery sequences, the stationary files use controlled purge phases while the raster uses open-air movement, the source coordinates are approximate, and height qualification retains **{len(raster_qc)}/{len(raster_window)} ({len(raster_qc)/len(raster_window):.1%})** reviewed readings.

The next independent validation should load frozen artifact **temporal-mint-seeker-v1** unchanged and apply it to new mint shapes and a randomized-order crossing experiment. Any retraining requires a new version. That prevents another round of tuning from turning Horizontal line raster-454 into its own answer key.
"""))



## Interpretation

Complete-trial nested validation improves balanced state accuracy from **83.9%** for a current-only snapshot to **87.7%** with causal temporal history. The locked deployment configuration is **Differences + recovery traces** with regularization `C=0.3`. This choice was made without consulting the raster.

On **Horizontal line raster-454**, the original classifier's approximate source-versus-outside medians are **0.979** versus **0.851**. After temporal logic plus the frozen odor-presence gate, active-mint medians are **1.000** versus **0.001**. The fraction of outside readings above 0.5 changes from **60.5%** to **22.2%**, while source-ranking AUC changes from **0.672** to **0.938**. Score-time correlation changes from **+0.53** to **+0.28**.

**Decision:** the temporal model is useful only if it improves complete-trial state separation and produces a more source-aligned active score without being selected against this map. The current experiment remains a pilot because there are only 15 independent mint recovery sequences, the stationary files use controlled purge phases while the raster uses open-air movement, the source coordinates are approximate, and height qualification retains **97/157 (61.8%)** reviewed readings.

The next independent validation should load frozen artifact **temporal-mint-seeker-v1** unchanged and apply it to new mint shapes and a randomized-order crossing experiment. Any retraining requires a new version. That prevents another round of tuning from turning Horizontal line raster-454 into its own answer key.
